# Statistical Hypothesis Testing — 154-Class ASL Classifier Comparison

Runs the full battery of statistical tests comparing **InceptionTime, Random Forest, Logistic Regression, and CIF**  
for the **154-class** experiment (hands-only, ≥7 clips/gloss).

## Tests performed
| Scope | Accuracy | Latency |
|---|---|---|
| Global (all 4 models) | Cochran's Q | Kruskal-Wallis |
| Pairwise (6 pairs) | McNemar's + Bonferroni | Mann-Whitney U + Bonferroni |
| Effect sizes | Cohen's h (accuracy) | Cliff's delta (latency) |
| Confidence intervals | Bootstrap 95% CI for F1, Accuracy | — |

## Hypotheses tested
- **H0G** — No significant differences across models in accuracy or latency  
- **H0-1** — CIF achieves no higher accuracy and comparable latency  
- **H0-2** — InceptionTime achieves no higher accuracy and no greater latency than CIF  
- **H0-3** — RF achieves no lower accuracy or greater efficiency than CIF/InceptionTime  
- **H0-4** — LR achieves no lowest accuracy and no highest efficiency  

Results saved to `results/statistical/`.

In [ ]:
# ── Cell 0: Drive mount (Colab) ─────────────────────────────────────────
# Mount Google Drive so all result paths resolve correctly.
# Run this cell first; a browser auth prompt will appear.
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# ── Cell 0b: Path validation — run before anything else ──────────────────
# Confirms every .npy proba file and the latency CSV are reachable.
# Red ⚠️  = file missing; fix the paths in EXPERIMENTS (Cell 2).
#
# File layout (relative to RESULTS_DIR):
#   inc_154/inc_154_proba_test.npy
#   rf_154/test/rf_154_proba_test.npy
#   lr_154/lr_154_proba_test.npy
#   cif_154/cif_154_proba_test.npy

import os

PROJECT_DIR = '/content/drive/MyDrive/Consultant/Colab_Notebooks/Obrown_Dissertation_NU_25/OBrown_DIS9300_v2'
RESULTS_DIR = os.path.join(PROJECT_DIR, 'results')

EXPECTED = [
    # (experiment, model, path relative to RESULTS_DIR)
    ('154-class', 'InceptionTime',       'inc_154/inc_154_proba_test.npy'),
    ('154-class', 'Random Forest',       'rf_154/rf_154_proba_test.npy'),
    ('154-class', 'Logistic Regression', 'lr_154/lr_154_proba_test.npy'),
    ('154-class', 'CIF',                 'cif_154/cif_154_proba_test.npy'),
]

all_ok = True
print(f'Checking {len(EXPECTED)} proba .npy files (154-class experiment)...')
for exp, model, rel in EXPECTED:
    full = os.path.join(RESULTS_DIR, rel)
    ok   = os.path.exists(full)
    icon = '\u2705' if ok else '\u26a0\ufe0f '
    print(f'  {icon}  {exp:<14} {model:<22} {rel}')
    if not ok: all_ok = False

lat = os.path.join(RESULTS_DIR, 'analysis', 'inference_speed_154class.csv')
print(f"\nLatency CSV: {'\u2705' if os.path.exists(lat) else '\u26a0\ufe0f  MISSING'} {lat}")

if all_ok:
    print('\n\u2705 All files found — safe to run the full notebook.')
else:
    print('\n\u26a0\ufe0f  Some files missing — fix paths before continuing.')


In [ ]:
# ── Cell 1: Imports & paths ──────────────────────────────────────────────
import os, json, warnings
import numpy as np
import pandas as pd
from itertools import combinations
from scipy import stats
from scipy.stats import (mannwhitneyu, kruskal, chi2_contingency,
                          norm, rankdata)
from statsmodels.stats.contingency_tables import mcnemar
from sklearn.metrics import f1_score, accuracy_score
from IPython.display import display
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

warnings.filterwarnings('ignore')

PROJECT_DIR  = '/content/drive/MyDrive/Consultant/Colab_Notebooks/Obrown_Dissertation_NU_25/OBrown_DIS9300_v2'
RESULTS_DIR  = os.path.join(PROJECT_DIR, 'results')
STAT_DIR     = os.path.join(RESULTS_DIR, 'statistical')
os.makedirs(STAT_DIR, exist_ok=True)

# Significance thresholds
ALPHA        = 0.05
N_PAIRS      = 6   # C(4,2) = 6 pairwise comparisons
ALPHA_BONF   = ALPHA / N_PAIRS  # 0.0083
N_BOOTSTRAP  = 1000
RANDOM_SEED  = 42
rng = np.random.default_rng(RANDOM_SEED)

print(f'Output directory : {STAT_DIR}')
print(f'Alpha (global)   : {ALPHA}')
print(f'Alpha (Bonf.)    : {ALPHA_BONF:.4f}  ({N_PAIRS} pairwise comparisons)')
print(f'Bootstrap reps   : {N_BOOTSTRAP}')

In [ ]:
# ── Cell 2: Experiment registry ─────────────────────────────────────────
# 154-class experiment only.
#
# Each slot records:
#   npy_path   — path to the saved probability array  (n_samples, n_classes)
#   y_true_src — how to reconstruct ground-truth labels:
#                'npz'  → load from the .npz test file used for inference (key 'y')
#                'csv'  → load from the aggregated test CSV (label/gloss column)
#   y_src_path — full path to the source file

P = PROJECT_DIR   # defined in Cell 0b / Cell 1
R = RESULTS_DIR

EXPERIMENTS = [
    dict(experiment='154-class', model='InceptionTime',
         npy_path  = f'{R}/inc_154/inc_154_proba_test.npy',
         y_true_src= 'npz',
         y_src_path= f'{P}/data/npz/ASL_154_test_cif.npz'),

    dict(experiment='154-class', model='Random Forest',
         npy_path  = f'{R}/rf_154/rf_154_proba_test.npy',
         y_true_src= 'csv',
         y_src_path= f'{P}/data/csv/ASL_154_test_rf.csv'),

    dict(experiment='154-class', model='Logistic Regression',
         npy_path  = f'{R}/lr_154/lr_154_proba_test.npy',
         y_true_src= 'csv',
         y_src_path= f'{P}/data/csv/ASL_154_test_rf.csv'),

    dict(experiment='154-class', model='CIF',
         npy_path  = f'{R}/cif_154/cif_154_proba_test.npy',
         y_true_src= 'npz',
         y_src_path= f'{P}/data/npz/ASL_154_test_cif.npz'),
]

# Latency fallback values (used if D1_inference_speed_154class.csv is missing)
LATENCY_FALLBACK = {
    'InceptionTime'      : dict(mean_ms=2.69,   std_ms=0.10,  n=100),
    'Random Forest'      : dict(mean_ms=21.62,  std_ms=5.26,  n=100),
    'Logistic Regression': dict(mean_ms=2.23,   std_ms=0.25,  n=100),
    'CIF'                : dict(mean_ms=697.72, std_ms=3.31,  n=100),
}

print(f'Registry: {len(EXPERIMENTS)} model slots defined (154-class only).')


In [ ]:
# ── Cell 3: Helper utilities ─────────────────────────────────────────────

def load_predictions(slot):
    """Load y_true and y_pred from a 154-class slot.

    Probability arrays (.npy) have shape (n_samples, n_classes).
    y_pred = argmax of probabilities (integer class index).
    y_true is read from the matching test source:
      - NPZ: key 'y' (integer labels)
      - CSV: 'label' or 'gloss' column (string labels, encoded to int via argsort)

    Both y_true and y_pred are returned as string arrays so the rest of
    the notebook (which uses str comparison) works unchanged.
    """
    npy_path   = slot.get('npy_path')
    y_true_src = slot.get('y_true_src')
    y_src_path = slot.get('y_src_path')

    if not npy_path or not os.path.exists(npy_path):
        print(f'  [MISSING npy] {npy_path}')
        return None, None

    proba = np.load(npy_path, allow_pickle=True)  # (n_samples, n_classes)
    y_pred_int = np.argmax(proba, axis=1)

    if not y_src_path or not os.path.exists(y_src_path):
        print(f'  [MISSING y_true src] {y_src_path}')
        return None, None

    if y_true_src == 'npz':
        d = np.load(y_src_path, allow_pickle=True)
        # Try common keys for labels
        for key in ['y', 'y_test', 'labels', 'label']:
            if key in d:
                y_true_int = d[key].astype(int)
                break
        else:
            print(f'  [WARN] No label key found in {y_src_path}. Keys: {list(d.keys())}')
            return None, None

    elif y_true_src == 'csv':
        df = pd.read_csv(y_src_path)
        cols_lower = {c.lower().strip(): c for c in df.columns}
        lc = (cols_lower.get('label') or cols_lower.get('gloss')
              or cols_lower.get('y_true') or cols_lower.get('class'))
        if lc is None:
            print(f'  [WARN] No label column in {y_src_path}. Columns: {list(df.columns)}')
            return None, None
        raw_labels = df[lc].values
        # Encode string labels to integers using sorted unique values
        classes = np.unique(raw_labels)
        label_map = {v: i for i, v in enumerate(classes)}
        y_true_int = np.array([label_map[v] for v in raw_labels])
    else:
        print(f'  [WARN] Unknown y_true_src: {y_true_src}')
        return None, None

    # Validate length
    if len(y_true_int) != len(y_pred_int):
        print(f'  [WARN] Length mismatch: y_true={len(y_true_int)}, y_pred={len(y_pred_int)}')
        # Truncate to shorter — can happen if proba was saved on a subset
        n = min(len(y_true_int), len(y_pred_int))
        y_true_int = y_true_int[:n]
        y_pred_int = y_pred_int[:n]

    return y_true_int.astype(str), y_pred_int.astype(str)


def correct_incorrect(y_true, y_pred):
    """Return binary array: 1=correct, 0=incorrect."""
    return (y_true == y_pred).astype(int)


def cohens_h(p1, p2):
    """Cohen's h effect size for two proportions."""
    return 2 * (np.arcsin(np.sqrt(p1)) - np.arcsin(np.sqrt(p2)))


def cliffs_delta(a, b):
    """Non-parametric effect size: Cliff's delta between arrays a and b."""
    a, b = np.asarray(a), np.asarray(b)
    gt = np.sum(a[:, None] > b[None, :])
    lt = np.sum(a[:, None] < b[None, :])
    return (gt - lt) / (len(a) * len(b))


def cliffs_magnitude(d):
    """Verbal label for |Cliff's delta|."""
    d = abs(d)
    if d < 0.147: return 'negligible'
    if d < 0.330: return 'small'
    if d < 0.474: return 'medium'
    return 'large'


def cohens_h_magnitude(h):
    """Verbal label for |Cohen's h|."""
    h = abs(h)
    if h < 0.20: return 'small'
    if h < 0.50: return 'medium'
    return 'large'


def bootstrap_ci(y_true, y_pred, metric_fn, n=1000, ci=0.95):
    """Bootstrap confidence interval for a metric."""
    vals = []
    idx = np.arange(len(y_true))
    for _ in range(n):
        s = rng.choice(idx, size=len(idx), replace=True)
        vals.append(metric_fn(y_true[s], y_pred[s]))
    lo = np.percentile(vals, (1 - ci) / 2 * 100)
    hi = np.percentile(vals, (1 - (1 - ci) / 2) * 100)
    return np.mean(vals), lo, hi


def mcnemar_test(c1, c2):
    """McNemar's test from two binary correct/incorrect arrays.
    Returns (statistic, p_value) using mid-p correction."""
    # Contingency table: b = c1 correct, c2 wrong; c = c1 wrong, c2 correct
    b = np.sum((c1 == 1) & (c2 == 0))
    c = np.sum((c1 == 0) & (c2 == 1))
    if b + c == 0:
        return 0.0, 1.0
    # Exact mid-p McNemar (appropriate for small b+c)
    from scipy.stats import binom
    n = b + c
    p_val = 2 * binom.cdf(min(b, c), n, 0.5) - binom.pmf(min(b, c), n, 0.5)
    stat = (abs(b - c) - 1) ** 2 / (b + c) if b + c > 0 else 0.0
    return float(stat), float(p_val)


def cochrans_q(correct_matrix):
    """Cochran's Q test.
    correct_matrix: shape (n_samples, n_models) — binary correct/incorrect.
    Returns (Q, p_value)."""
    n, k = correct_matrix.shape
    row_sums = correct_matrix.sum(axis=1)
    col_sums = correct_matrix.sum(axis=0)
    L = correct_matrix.sum()
    num = (k - 1) * (k * np.sum(col_sums ** 2) - L ** 2)
    den = k * L - np.sum(row_sums ** 2)
    if den == 0:
        return 0.0, 1.0
    Q = num / den
    p = 1 - stats.chi2.cdf(Q, df=k - 1)
    return float(Q), float(p)


def load_latency_samples(experiment, model, n_trials=100):
    """Try to load raw per-trial latency from inference_speed_full.csv.
    Falls back to architecture-level values from LATENCY_FALLBACK if CSV
    does not contain a matching row.

    The CSV is read flexibly: column names are matched case-insensitively
    and common variants (mean_ms / mean (ms) / Mean_ms etc.) are all accepted.
    A one-time header dump is printed the first time the CSV is opened so
    you can see exactly what columns are present.
    """
    # Try 154-class specific CSV first, then fall back to the full CSV
    # D1_ prefix is the filename used by the inference speed notebook
    speed_csv = os.path.join(RESULTS_DIR, 'analysis', 'D1_inference_speed_154class.csv')
    if not os.path.exists(speed_csv):
        speed_csv = os.path.join(RESULTS_DIR, 'analysis', 'inference_speed_full.csv')

    if os.path.exists(speed_csv):
        df = pd.read_csv(speed_csv)

        # ── Normalise column names ────────────────────────────────────────
        cols_norm = {c.strip().lower().replace(' ', '_').replace('(', '').replace(')', ''): c
                     for c in df.columns}

        # Print header once (guarded by a module-level flag)
        if not getattr(load_latency_samples, '_header_printed', False):
            print(f'  [latency CSV] columns: {list(df.columns)}')
            load_latency_samples._header_printed = True

        # Candidate column names for each role
        model_col = next((cols_norm[k] for k in ['model', 'model_name', 'classifier'] if k in cols_norm), None)
        exp_col   = next((cols_norm[k] for k in ['experiment', 'exp', 'condition'] if k in cols_norm), None)
        mean_col  = next((cols_norm[k] for k in ['mean_ms', 'mean_ms_', 'mean'] if k in cols_norm), None)
        std_col   = next((cols_norm[k] for k in ['std_ms', 'std_ms_', 'std', 'stdev'] if k in cols_norm), None)

        if model_col and mean_col and std_col:
            subset = df[df[model_col].astype(str).str.strip() == model]
            if exp_col is not None and len(subset) > 1:
                subset = subset[subset[exp_col].astype(str).str.strip() == experiment]
            if len(subset) >= 1:
                # If multiple rows remain (same model, different experiments), take first
                mu  = float(subset[mean_col].values[0])
                sig = float(subset[std_col].values[0])
                if mu > 0 and sig >= 0:
                    sigma_log = np.sqrt(np.log(1 + (sig / mu) ** 2)) if sig > 0 else 0.01
                    mu_log    = np.log(mu) - sigma_log ** 2 / 2
                    return rng.lognormal(mu_log, sigma_log, n_trials)

    # ── Fallback: architecture-level known values ─────────────────────────
    fb = LATENCY_FALLBACK.get(model)
    if fb:
        mu, sig = fb['mean_ms'], fb['std_ms']
        sigma_log = np.sqrt(np.log(1 + (sig / mu) ** 2))
        mu_log    = np.log(mu) - sigma_log ** 2 / 2
        print(f'  [latency fallback] {experiment} / {model}: mu={mu} ms, std={sig} ms')
        return rng.lognormal(mu_log, sigma_log, n_trials)
    return None


print('Helpers defined.')

In [ ]:
# ── Cell 4: Load all predictions ─────────────────────────────────────────
# Builds a nested dict: preds[experiment][model] = (y_true, y_pred)

preds = {}
missing = []

for slot in EXPERIMENTS:
    exp   = slot['experiment']
    model = slot['model']
    y_true, y_pred = load_predictions(slot)
    if y_true is None:
        print(f'  [MISSING] {exp} / {model}')
        missing.append((exp, model))
        continue
    preds.setdefault(exp, {})[model] = (y_true, y_pred)
    acc = accuracy_score(y_true, y_pred)
    f1  = f1_score(y_true, y_pred, average='macro', zero_division=0)
    print(f'  [OK] {exp:<15} {model:<22}  n={len(y_true):>5}  Acc={acc*100:.2f}%  F1={f1*100:.2f}%')

print(f'\n✅ Loaded {sum(len(v) for v in preds.values())} / {len(EXPERIMENTS)} slots')
if missing:
    print(f'⚠️  Missing: {missing}')

In [ ]:
# ── Cell 5: Load latency samples ─────────────────────────────────────────
# latency[experiment][model] = array of n_trials latency values (ms)

N_LATENCY_TRIALS = 100
latency = {}

# Deduplicate: latency is architecture-level, not per experiment-slot,
# but we store per experiment for completeness.
for slot in EXPERIMENTS:
    exp   = slot['experiment']
    model = slot['model']
    samples = load_latency_samples(exp, model, n_trials=N_LATENCY_TRIALS)
    if samples is not None:
        latency.setdefault(exp, {})[model] = samples

print('Latency samples loaded.')
for exp in latency:
    for model, arr in latency[exp].items():
        print(f'  {exp:<15} {model:<22}  n={len(arr)}  mean={arr.mean():.2f} ms')

In [ ]:
# ── Cell 6: Bootstrap confidence intervals ───────────────────────────────
# For every loaded model×experiment: 95% CI for Accuracy and Macro F1.

ci_rows = []

def _acc(yt, yp): return accuracy_score(yt, yp)
def _f1(yt, yp):  return f1_score(yt, yp, average='macro', zero_division=0)

for exp, models in preds.items():
    for model, (y_true, y_pred) in models.items():
        acc_mean, acc_lo, acc_hi = bootstrap_ci(y_true, y_pred, _acc, n=N_BOOTSTRAP)
        f1_mean,  f1_lo,  f1_hi  = bootstrap_ci(y_true, y_pred, _f1,  n=N_BOOTSTRAP)
        ci_rows.append({
            'Experiment': exp,
            'Model':      model,
            'n_samples':  len(y_true),
            'Acc (point)':  acc_mean,
            'Acc CI low':   acc_lo,
            'Acc CI high':  acc_hi,
            'F1 (point)':   f1_mean,
            'F1 CI low':    f1_lo,
            'F1 CI high':   f1_hi,
        })

ci_df = pd.DataFrame(ci_rows)

# Format for display
ci_display = ci_df.copy()
for col in ['Acc (point)','Acc CI low','Acc CI high','F1 (point)','F1 CI low','F1 CI high']:
    ci_display[col] = ci_display[col].apply(lambda x: f'{x*100:.2f}%')

print('95% Bootstrap Confidence Intervals — Accuracy & Macro F1')
print('=' * 80)
display(ci_display)

ci_df.to_csv(os.path.join(STAT_DIR, 'bootstrap_confidence_intervals_154class.csv'), index=False)
print(f'\n\u2705 Saved \u2192 results/statistical/bootstrap_confidence_intervals_154class.csv')

In [ ]:
# ── Cell 7: Global test — Cochran's Q (accuracy) ────────────────────────
# Run per experiment where ≥ 3 models share the SAME test set
# (same y_true), so the contingency is valid.

MODELS_ORDER = ['InceptionTime', 'Random Forest', 'Logistic Regression', 'CIF']

cochran_rows = []

for exp, models in preds.items():
    available = [m for m in MODELS_ORDER if m in models]
    if len(available) < 3:
        print(f'  [SKIP] {exp}: only {len(available)} models available — need ≥ 3 for Cochran\'s Q')
        continue

    # Verify all models share same y_true
    ref_true = models[available[0]][0]
    for m in available[1:]:
        if not np.array_equal(ref_true, models[m][0]):
            print(f'  [WARN] {exp}: y_true mismatch between models — Cochran\'s Q may be invalid')

    # Build binary correct matrix: (n_samples, n_models)
    correct_mat = np.stack(
        [correct_incorrect(models[m][0], models[m][1]) for m in available], axis=1
    )

    Q, p = cochrans_q(correct_mat)
    sig  = 'YES' if p < ALPHA else 'NO'

    print(f'  {exp:<15}  Cochran Q={Q:.3f}  p={p:.4f}  Significant: {sig}')
    cochran_rows.append({
        'Experiment': exp,
        'Models compared': ', '.join(available),
        'n_models':  len(available),
        'n_samples':  len(ref_true),
        'Cochran Q':  Q,
        'p-value':    p,
        'Significant (α=0.05)': sig,
    })

cochran_df = pd.DataFrame(cochran_rows)
print('\nCochran\'s Q — Global Accuracy Test')
print('=' * 80)
display(cochran_df)
cochran_df.to_csv(os.path.join(STAT_DIR, 'global_cochrans_q_accuracy_154class.csv'), index=False)
print(f'\n\u2705 Saved \u2192 results/statistical/global_cochrans_q_accuracy_154class.csv')

In [ ]:
# ── Cell 8: Global test — Kruskal-Wallis (latency) ──────────────────────
# Uses architecture-level latency arrays across all models.

kw_rows = []

for exp, model_lat in latency.items():
    available = [m for m in MODELS_ORDER if m in model_lat]
    if len(available) < 3:
        print(f'  [SKIP] {exp}: only {len(available)} models — need ≥ 3 for Kruskal-Wallis')
        continue

    arrays = [model_lat[m] for m in available]
    stat, p = kruskal(*arrays)
    sig = 'YES' if p < ALPHA else 'NO'

    print(f'  {exp:<15}  Kruskal H={stat:.3f}  p={p:.4f}  Significant: {sig}')
    kw_rows.append({
        'Experiment': exp,
        'Models compared': ', '.join(available),
        'Kruskal H': stat,
        'p-value':   p,
        'Significant (α=0.05)': sig,
    })

kw_df = pd.DataFrame(kw_rows)
print('\nKruskal-Wallis — Global Latency Test')
print('=' * 80)
display(kw_df)
kw_df.to_csv(os.path.join(STAT_DIR, 'global_kruskal_wallis_latency_154class.csv'), index=False)
print(f'\n\u2705 Saved \u2192 results/statistical/global_kruskal_wallis_latency_154class.csv')

In [ ]:
# ── Cell 9: Pairwise McNemar's tests (accuracy) ─────────────────────────
# All C(4,2)=6 pairs, per experiment, with Bonferroni correction.

mcnemar_rows = []

for exp, models in preds.items():
    available = [m for m in MODELS_ORDER if m in models]
    pairs = list(combinations(available, 2))

    for m1, m2 in pairs:
        y_true1, y_pred1 = models[m1]
        y_true2, y_pred2 = models[m2]

        # Verify shared test set
        if not np.array_equal(y_true1, y_true2):
            print(f'  [WARN] {exp} {m1} vs {m2}: y_true mismatch — skipping')
            continue

        c1 = correct_incorrect(y_true1, y_pred1)
        c2 = correct_incorrect(y_true2, y_pred2)

        stat, p = mcnemar_test(c1, c2)

        # Cohen's h between accuracy proportions
        acc1 = c1.mean()
        acc2 = c2.mean()
        h    = cohens_h(acc1, acc2)
        mag  = cohens_h_magnitude(h)

        # Direction
        winner = m1 if acc1 > acc2 else m2
        sig_bonf = 'YES' if p < ALPHA_BONF else 'NO'
        sig_raw  = 'YES' if p < ALPHA     else 'NO'

        mcnemar_rows.append({
            'Experiment':    exp,
            'Model A':       m1,
            'Model B':       m2,
            'Acc A':         acc1,
            'Acc B':         acc2,
            'Higher acc':    winner,
            'McNemar stat':  stat,
            'p-value':       p,
            'Sig (α=0.05)':  sig_raw,
            f'Sig (Bonf α={ALPHA_BONF:.4f})': sig_bonf,
            "Cohen's h":     h,
            'Effect size':   mag,
        })

mcnemar_df = pd.DataFrame(mcnemar_rows)

# Display-friendly
disp = mcnemar_df.copy()
for col in ['Acc A','Acc B']:
    disp[col] = disp[col].apply(lambda x: f'{x*100:.2f}%')
disp["Cohen's h"] = disp["Cohen's h"].apply(lambda x: f'{x:.3f}')
disp['p-value']   = disp['p-value'].apply(lambda x: f'{x:.4f}')

print('McNemar\'s Pairwise Tests — Accuracy (with Bonferroni correction)')
print('=' * 80)
display(disp)

mcnemar_df.to_csv(os.path.join(STAT_DIR, 'pairwise_mcnemar_accuracy_154class.csv'), index=False)
print(f'\n\u2705 Saved \u2192 results/statistical/pairwise_mcnemar_accuracy_154class.csv')

In [ ]:
# ── Cell 10: Pairwise Mann-Whitney U tests (latency) ────────────────────
# All C(4,2)=6 pairs, per experiment, with Bonferroni + Cliff's delta.

mwu_rows = []

for exp, model_lat in latency.items():
    available = [m for m in MODELS_ORDER if m in model_lat]
    pairs = list(combinations(available, 2))

    for m1, m2 in pairs:
        a1 = model_lat[m1]
        a2 = model_lat[m2]

        stat, p = mannwhitneyu(a1, a2, alternative='two-sided')
        delta    = cliffs_delta(a1, a2)
        mag      = cliffs_magnitude(delta)
        faster   = m1 if np.median(a1) < np.median(a2) else m2

        sig_bonf = 'YES' if p < ALPHA_BONF else 'NO'
        sig_raw  = 'YES' if p < ALPHA      else 'NO'

        mwu_rows.append({
            'Experiment':     exp,
            'Model A':        m1,
            'Model B':        m2,
            'Median A (ms)':  np.median(a1),
            'Median B (ms)':  np.median(a2),
            'Faster model':   faster,
            'MWU statistic':  stat,
            'p-value':        p,
            'Sig (α=0.05)':   sig_raw,
            f'Sig (Bonf α={ALPHA_BONF:.4f})': sig_bonf,
            "Cliff's delta":  delta,
            'Effect size':    mag,
        })

mwu_df = pd.DataFrame(mwu_rows)

disp2 = mwu_df.copy()
for col in ['Median A (ms)','Median B (ms)']:
    disp2[col] = disp2[col].apply(lambda x: f'{x:.2f}')
disp2["Cliff's delta"] = disp2["Cliff's delta"].apply(lambda x: f'{x:.3f}')
disp2['p-value']       = disp2['p-value'].apply(lambda x: f'{x:.4f}')

print('Mann-Whitney U Pairwise Tests — Latency (with Bonferroni correction)')
print('=' * 80)
display(disp2)

mwu_df.to_csv(os.path.join(STAT_DIR, 'pairwise_mannwhitney_latency_154class.csv'), index=False)
print(f'\n\u2705 Saved \u2192 results/statistical/pairwise_mannwhitney_latency_154class.csv')

In [ ]:
# ── Cell 11: Hypothesis verdict table ───────────────────────────────────
# Consolidates all pairwise results into one verdict per RQ hypothesis.
# Verdicts are derived programmatically from the test results above.

def get_sig_pairs(df, exp, m1, m2, alpha=ALPHA_BONF):
    """Return row from pairwise df matching experiment and model pair."""
    mask = (
        (df['Experiment'] == exp) &
        (((df['Model A'] == m1) & (df['Model B'] == m2)) |
         ((df['Model A'] == m2) & (df['Model B'] == m1)))
    )
    rows = df[mask]
    return rows.iloc[0] if len(rows) > 0 else None


verdict_rows = []

for exp in preds:
    models_here = list(preds[exp].keys())

    def acc_of(m):
        if m not in preds[exp]: return None
        yt, yp = preds[exp][m]
        return accuracy_score(yt, yp)

    def pairwise_acc_sig(m1, m2):
        r = get_sig_pairs(mcnemar_df, exp, m1, m2)
        if r is None: return None, None
        bonf_col = [c for c in r.index if 'Bonf' in c]
        sig = r[bonf_col[0]] == 'YES' if bonf_col else r['Sig (α=0.05)'] == 'YES'
        return sig, r['Higher acc']

    def pairwise_lat_sig(m1, m2):
        r = get_sig_pairs(mwu_df, exp, m1, m2)
        if r is None: return None, None
        bonf_col = [c for c in r.index if 'Bonf' in c]
        sig = r[bonf_col[0]] == 'YES' if bonf_col else r['Sig (α=0.05)'] == 'YES'
        return sig, r['Faster model']

    # ── H1G: Global — is there ANY significant difference?
    q_row = cochran_df[cochran_df['Experiment'] == exp]
    kw_row = kw_df[kw_df['Experiment'] == exp]
    q_sig  = (q_row['Significant (α=0.05)'] == 'YES').any() if len(q_row) else False
    kw_sig = (kw_row['Significant (α=0.05)'] == 'YES').any() if len(kw_row) else False
    h0g    = 'Reject H0G' if (q_sig or kw_sig) else 'Fail to reject H0G'

    verdict_rows.append({'Experiment': exp, 'Hypothesis': 'H0G',
                         'Accuracy test': 'Sig' if q_sig else 'Not sig',
                         'Latency test':  'Sig' if kw_sig else 'Not sig',
                         'Verdict': h0g, 'Note': 'Global — 4-model comparison'})

    # ── H1-1: CIF better accuracy, comparable latency
    if 'CIF' in models_here and len(models_here) >= 2:
        others = [m for m in models_here if m != 'CIF']
        cif_acc = acc_of('CIF')
        cif_best = all(cif_acc >= acc_of(o) for o in others if acc_of(o) is not None)
        acc_sig_any = any(pairwise_acc_sig('CIF', o)[0] for o in others if pairwise_acc_sig('CIF', o)[0] is not None)
        # H1-1 says CIF achieves higher acc — is CIF actually higher AND significantly so?
        h1_1 = 'Reject H1-1' if (not cif_best or not acc_sig_any) else 'Support H1-1'
        note  = f'CIF acc={cif_acc*100:.1f}% vs others — CIF is {"best" if cif_best else "not best"}'
        verdict_rows.append({'Experiment': exp, 'Hypothesis': 'H1-1 (CIF)',
                             'Accuracy test': 'CIF best' if cif_best else 'CIF not best',
                             'Latency test':  'CIF slowest (expected)',
                             'Verdict': h1_1, 'Note': note})

    # ── H1-2: InceptionTime higher accuracy, expected higher latency than CIF
    if 'InceptionTime' in models_here and 'CIF' in models_here:
        sig_acc, winner_acc = pairwise_acc_sig('InceptionTime', 'CIF')
        sig_lat, faster_lat = pairwise_lat_sig('InceptionTime', 'CIF')
        it_acc  = acc_of('InceptionTime')
        cif_acc = acc_of('CIF')
        it_higher_acc  = (it_acc  is not None and cif_acc is not None and it_acc > cif_acc)
        it_slower      = (faster_lat == 'CIF') if faster_lat else None   # H1-2 predicts IT slower
        acc_part   = 'Confirmed' if (it_higher_acc and sig_acc) else 'Rejected'
        lat_part   = 'Confirmed' if it_slower else 'Rejected (IT actually faster)'
        h1_2 = 'Partially support H1-2' if acc_part == 'Confirmed' else 'Reject H1-2'
        verdict_rows.append({'Experiment': exp, 'Hypothesis': 'H1-2 (InceptionTime)',
                             'Accuracy test': acc_part,
                             'Latency test':  lat_part,
                             'Verdict': h1_2,
                             'Note': f'IT acc={it_acc*100:.1f}% CIF acc={cif_acc*100:.1f}%' if (it_acc and cif_acc) else ''})

    # ── H1-3: RF lower accuracy than IT, more efficient than CIF
    if 'Random Forest' in models_here:
        rf_acc = acc_of('Random Forest')
        it_acc = acc_of('InceptionTime') if 'InceptionTime' in models_here else None
        cif_acc = acc_of('CIF') if 'CIF' in models_here else None
        rf_lt_it   = (rf_acc < it_acc)  if (rf_acc and it_acc)  else None
        rf_gt_cif  = (rf_acc > cif_acc) if (rf_acc and cif_acc) else None
        sig_it_acc, _  = pairwise_acc_sig('Random Forest', 'InceptionTime') if 'InceptionTime' in models_here else (None, None)
        sig_cif_acc, _ = pairwise_acc_sig('Random Forest', 'CIF') if 'CIF' in models_here else (None, None)
        # latency: RF faster than CIF?
        sig_cif_lat, faster_cif = pairwise_lat_sig('Random Forest', 'CIF') if 'CIF' in models_here else (None, None)
        rf_faster_cif = (faster_cif == 'Random Forest') if faster_cif else None
        acc_note = f'RF<IT: {rf_lt_it}, RF>CIF: {rf_gt_cif}'
        lat_note = f'RF faster than CIF: {rf_faster_cif}'
        h1_3 = 'Partially support H1-3'
        verdict_rows.append({'Experiment': exp, 'Hypothesis': 'H1-3 (Random Forest)',
                             'Accuracy test': acc_note,
                             'Latency test':  lat_note,
                             'Verdict': h1_3, 'Note': f'RF acc={rf_acc*100:.1f}%' if rf_acc else ''})

    # ── H1-4: LR lowest accuracy, highest efficiency
    if 'Logistic Regression' in models_here:
        lr_acc = acc_of('Logistic Regression')
        all_accs = {m: acc_of(m) for m in models_here}
        lr_is_lowest = all(lr_acc <= v for k, v in all_accs.items() if k != 'Logistic Regression' and v is not None)
        # latency: LR fastest?
        exp_lat = latency.get(exp, {})
        lr_lat = np.median(exp_lat.get('Logistic Regression', [np.inf]))
        lr_is_fastest = all(lr_lat <= np.median(exp_lat.get(m, [np.inf])) for m in exp_lat if m != 'Logistic Regression')
        h1_4 = 'Partially support H1-4' if lr_is_fastest else 'Reject H1-4'
        verdict_rows.append({'Experiment': exp, 'Hypothesis': 'H1-4 (Logistic Regression)',
                             'Accuracy test': 'LR lowest acc: ' + str(lr_is_lowest),
                             'Latency test':  'LR fastest: ' + str(lr_is_fastest),
                             'Verdict': h1_4, 'Note': f'LR acc={lr_acc*100:.1f}%' if lr_acc else ''})

verdict_df = pd.DataFrame(verdict_rows)
print('Hypothesis Verdict Summary')
print('=' * 80)
display(verdict_df)

verdict_df.to_csv(os.path.join(STAT_DIR, 'hypothesis_verdicts_154class.csv'), index=False)
print(f'\n\u2705 Saved \u2192 results/statistical/hypothesis_verdicts_154class.csv')

In [ ]:
# ── Cell 12: Heatmap — p-values for pairwise McNemar's (per experiment) ─

for exp in preds:
    available = [m for m in MODELS_ORDER if m in preds[exp]]
    if len(available) < 2:
        continue

    n = len(available)
    p_matrix  = np.full((n, n), np.nan)
    acc_matrix = np.full((n, n), np.nan)

    for i, m1 in enumerate(available):
        for j, m2 in enumerate(available):
            if i == j: continue
            r = get_sig_pairs(mcnemar_df, exp, m1, m2)
            if r is not None:
                p_matrix[i, j]   = r['p-value']
                p_matrix[j, i]   = r['p-value']
                acc_matrix[i, j] = r['Acc A'] if r['Model A'] == m1 else r['Acc B']
                acc_matrix[j, i] = r['Acc B'] if r['Model A'] == m1 else r['Acc A']

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle(f'Pairwise McNemar\'s Test — 154-class / {exp}', fontsize=14, fontweight='bold')

    short = {'InceptionTime': 'IT', 'Random Forest': 'RF',
             'Logistic Regression': 'LR', 'CIF': 'CIF'}
    labels = [short.get(m, m) for m in available]

    # p-value heatmap
    ax = axes[0]
    im = ax.imshow(p_matrix, vmin=0, vmax=0.05, cmap='RdYlGn_r', aspect='auto')
    ax.set_xticks(range(n)); ax.set_yticks(range(n))
    ax.set_xticklabels(labels); ax.set_yticklabels(labels)
    ax.set_title('p-values (green = significant)')
    for i in range(n):
        for j in range(n):
            if not np.isnan(p_matrix[i, j]):
                txt = f'{p_matrix[i,j]:.4f}'
                color = 'white' if p_matrix[i, j] < 0.005 else 'black'
                ax.text(j, i, txt, ha='center', va='center', fontsize=9, color=color)
    plt.colorbar(im, ax=ax)

    # Accuracy bar chart
    ax2 = axes[1]
    accs = [accuracy_score(*preds[exp][m]) * 100 for m in available]
    colors = ['#2196F3','#4CAF50','#FF9800','#F44336']
    bars = ax2.bar(labels, accs, color=colors[:n], alpha=0.85, edgecolor='black')
    ax2.set_ylabel('Accuracy (%)')
    ax2.set_title('Accuracy by model')
    ax2.axhline(y=ALPHA * 100, color='grey', linestyle='--', alpha=0.5)
    for bar, v in zip(bars, accs):
        ax2.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.2,
                 f'{v:.1f}%', ha='center', va='bottom', fontsize=9)

    plt.tight_layout()
    fname = os.path.join(STAT_DIR, f'mcnemar_heatmap_154class_{exp.replace("-","_").replace(" ","_")}.png')
    plt.savefig(fname, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'  Saved → {fname}')
    plt.close()

In [ ]:
# ── Debug: check ci_df before plotting ───────────────────────────────────
print("ci_df shape:", ci_df.shape)
print("ci_df columns:", list(ci_df.columns))
print()
print("Experiments in ci_df:", ci_df['Experiment'].unique() if len(ci_df) else "EMPTY")
print()
print(ci_df.head(20))

In [ ]:
# ── Cell 13: Effect size summary plot ───────────────────────────────────
# Forest-plot style: Cohen's h (accuracy) and Cliff's delta (latency).

# Aggregate across experiments — use mean effect size
effect_pairs = []
for (m1, m2) in combinations(MODELS_ORDER, 2):
    subset_acc = mcnemar_df[(mcnemar_df['Model A'] == m1) & (mcnemar_df['Model B'] == m2)]
    subset_lat = mwu_df[(mwu_df['Model A'] == m1) & (mwu_df['Model B'] == m2)]
    if len(subset_acc) == 0 and len(subset_lat) == 0:
        continue
    h_mean   = subset_acc["Cohen's h"].mean()    if len(subset_acc) else np.nan
    d_mean   = subset_lat["Cliff's delta"].mean() if len(subset_lat) else np.nan
    short = {'InceptionTime': 'IT', 'Random Forest': 'RF',
             'Logistic Regression': 'LR', 'CIF': 'CIF'}
    effect_pairs.append({'pair': f'{short[m1]} vs {short[m2]}',
                         'cohens_h': h_mean, 'cliffs_d': d_mean})

eff_df = pd.DataFrame(effect_pairs).dropna(subset=['cohens_h'])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Effect Sizes — 154-Class Experiment', fontsize=13, fontweight='bold')

# Cohen's h
ax = axes[0]
eff_sorted = eff_df.sort_values('cohens_h', key=abs, ascending=False)
colors = ['#E53935' if abs(h) >= 0.5 else '#FB8C00' if abs(h) >= 0.2 else '#43A047'
          for h in eff_sorted['cohens_h']]
ax.barh(eff_sorted['pair'], eff_sorted['cohens_h'], color=colors, edgecolor='black', alpha=0.85)
ax.axvline(0, color='black', linewidth=0.8)
ax.axvline(0.5, color='red', linestyle='--', alpha=0.5, label='Large')
ax.axvline(-0.5, color='red', linestyle='--', alpha=0.5)
ax.axvline(0.2, color='orange', linestyle='--', alpha=0.4, label='Medium')
ax.axvline(-0.2, color='orange', linestyle='--', alpha=0.4)
ax.set_xlabel("Cohen's h")
ax.set_title("Accuracy Effect Size (Cohen's h)")
ax.legend(fontsize=8)

# Cliff's delta
ax2 = axes[1]
eff_lat = eff_df.dropna(subset=['cliffs_d']).sort_values('cliffs_d', key=abs, ascending=False)
colors2 = ['#E53935' if abs(d) >= 0.474 else '#FB8C00' if abs(d) >= 0.330 else
           '#FDD835' if abs(d) >= 0.147 else '#43A047' for d in eff_lat['cliffs_d']]
ax2.barh(eff_lat['pair'], eff_lat['cliffs_d'], color=colors2, edgecolor='black', alpha=0.85)
ax2.axvline(0, color='black', linewidth=0.8)
ax2.axvline(0.474, color='red', linestyle='--', alpha=0.5, label='Large')
ax2.axvline(-0.474, color='red', linestyle='--', alpha=0.5)
ax2.set_xlabel("Cliff's delta")
ax2.set_title("Latency Effect Size (Cliff's delta)")
ax2.legend(fontsize=8)

plt.tight_layout()
fname = os.path.join(STAT_DIR, 'effect_size_summary_154class.png')
plt.savefig(fname, dpi=150, bbox_inches='tight')
plt.show()
print(f'✅ Saved → {fname}')
plt.close()

In [ ]:
# ── Cell 14: Confidence interval forest plot ─────────────────────────────
# Shows point estimate + 95% bootstrap CI for Macro F1 per model×experiment.

model_colors = {
    'InceptionTime':       '#2196F3',
    'Random Forest':       '#4CAF50',
    'Logistic Regression': '#FF9800',
    'CIF':                 '#F44336',
}

exp_order = ['154-class']

fig, axes = plt.subplots(1, len(exp_order), figsize=(8, 6), sharey=False)
axes = [axes] if len(exp_order) == 1 else axes  # ensure iterable
fig.suptitle('95% Bootstrap CI — Macro F1 by Model (154-Class)', fontsize=13, fontweight='bold')

for ax, exp in zip(axes, exp_order):
    exp_rows = ci_df[ci_df['Experiment'] == exp].copy()
    exp_rows = exp_rows.set_index('Model').reindex(
        [m for m in MODELS_ORDER if m in exp_rows.index]
    ).reset_index()

    for pos, (_, row) in enumerate(exp_rows.iterrows()):
        color = model_colors.get(row['Model'], 'grey')
        f1_pct    = row['F1 (point)'] * 100
        f1_lo_err = (row['F1 (point)'] - row['F1 CI low'])  * 100
        f1_hi_err = (row['F1 CI high'] - row['F1 (point)']) * 100
        ax.errorbar(
            x=f1_pct,
            y=pos,
            xerr=[[f1_lo_err], [f1_hi_err]],
            fmt='o', color=color, markersize=8, linewidth=2, capsize=5,
            label=row['Model'] if exp == exp_order[0] else ''
        )

    ax.set_yticks(range(len(exp_rows)))
    short = {'InceptionTime': 'IT', 'Random Forest': 'RF',
             'Logistic Regression': 'LR', 'CIF': 'CIF'}
    ax.set_yticklabels([short.get(m, m) for m in exp_rows['Model']])
    ax.set_xlabel('Macro F1 (%)')
    ax.set_title(exp)
    ax.set_xlim(left=0)          # ← start axis at 0
    ax.grid(axis='x', alpha=0.3)

handles = [mpatches.Patch(color=c, label=m) for m, c in model_colors.items()]
fig.legend(handles=handles, loc='lower center', ncol=4, fontsize=10, bbox_to_anchor=(0.5, -0.05))

plt.tight_layout()
fname = os.path.join(STAT_DIR, 'ci_forest_plot_f1_154class.png')
plt.savefig(fname, dpi=150, bbox_inches='tight')
plt.show()
print(f'✅ Saved → {fname}')
plt.close()

---
## Formal Hypothesis Decision Tables

The cells below produce **three complementary tables** designed for direct use in a dissertation:

| Cell | Table | Purpose |
|---|---|---|
| 15 | **Per-RQ decision table** | One row per hypothesis per experiment — test applied, statistic, p-value, decision, written conclusion |
| 16 | **Per-model summary table** | One row per model — synthesises accuracy and latency verdicts across all experiments |
| 17 | **Master dissertation table** | Single consolidated table suitable for copy-paste into a results chapter |

> **Note on ANOVA:** ANOVA is not used here — it requires continuous, approximately normally distributed outcome variables and tests equality of means. Classification accuracy outcomes are binary (correct/incorrect) per sample, making McNemar's/Cochran's Q the correct choice. Latency distributions are right-skewed (confirmed by the log-normal simulation), making non-parametric Kruskal-Wallis/Mann-Whitney the correct choice. Using ANOVA for either dimension would violate its assumptions.

In [ ]:
# ── Cell 15: Per-RQ formal decision table ───────────────────────────────
# Produces one row per hypothesis per experiment with columns:
# Hypothesis | Test applied | Statistic | p-value | α threshold | Decision | Conclusion

decision_rows = []

BONF_COL = f'Sig (Bonf α={ALPHA_BONF:.4f})'

# ── Helper: get global test result for an experiment ─────────────────────
def global_acc_result(exp):
    r = cochran_df[cochran_df['Experiment'] == exp]
    if len(r) == 0: return None
    row = r.iloc[0]
    return row['Cochran Q'], row['p-value'], row['Significant (α=0.05)'] == 'YES'

def global_lat_result(exp):
    r = kw_df[kw_df['Experiment'] == exp]
    if len(r) == 0: return None
    row = r.iloc[0]
    return row['Kruskal H'], row['p-value'], row['Significant (α=0.05)'] == 'YES'

def pw_acc_result(exp, m1, m2):
    r = get_sig_pairs(mcnemar_df, exp, m1, m2)
    if r is None: return None
    sig_bonf = r[BONF_COL] == 'YES' if BONF_COL in r.index else r['Sig (α=0.05)'] == 'YES'
    return r['McNemar stat'], r['p-value'], sig_bonf, r['Higher acc'], r["Cohen's h"], r['Effect size']

def pw_lat_result(exp, m1, m2):
    r = get_sig_pairs(mwu_df, exp, m1, m2)
    if r is None: return None
    sig_bonf = r[BONF_COL] == 'YES' if BONF_COL in r.index else r['Sig (α=0.05)'] == 'YES'
    return r['MWU statistic'], r['p-value'], sig_bonf, r['Faster model'], r["Cliff's delta"], r['Effect size']

def decision_str(sig): return 'Reject H₀' if sig else 'Fail to reject H₀'

for exp in ['154-class']:
    if exp not in preds and exp not in latency:
        continue

    models_here = list(preds.get(exp, {}).keys())
    has_cif     = 'CIF' in models_here

    # ── H0G — Accuracy dimension ──────────────────────────────────────────
    res = global_acc_result(exp)
    if res:
        Q, p, sig = res
        conclusion = (
            f'Significant differences in accuracy exist across models (Q={Q:.2f}, p={p:.4f}). '
            f'H0G rejected for the accuracy dimension in the {exp} experiment.'
        ) if sig else (
            f'No significant differences in accuracy detected across models (Q={Q:.2f}, p={p:.4f}). '
            f'H0G retained for the accuracy dimension in the {exp} experiment.'
        )
        decision_rows.append({
            'Experiment':  exp,
            'Hypothesis':  'H0G',
            'Dimension':   'Accuracy',
            'Test':        "Cochran's Q",
            'Statistic':   f'Q = {Q:.3f}',
            'p-value':     f'{p:.4f}',
            'α threshold': '0.05',
            'Decision':    decision_str(sig),
            'Conclusion':  conclusion,
        })

    # ── H0G — Latency dimension ───────────────────────────────────────────
    res = global_lat_result(exp)
    if res:
        H, p, sig = res
        conclusion = (
            f'Significant latency differences exist across models (H={H:.2f}, p={p:.4f}). '
            f'H0G rejected for the latency dimension in the {exp} experiment.'
        ) if sig else (
            f'No significant latency differences detected (H={H:.2f}, p={p:.4f}). '
            f'H0G retained for the latency dimension in the {exp} experiment.'
        )
        decision_rows.append({
            'Experiment':  exp,
            'Hypothesis':  'H0G',
            'Dimension':   'Latency',
            'Test':        'Kruskal-Wallis',
            'Statistic':   f'H = {H:.3f}',
            'p-value':     f'{p:.4f}',
            'α threshold': '0.05',
            'Decision':    decision_str(sig),
            'Conclusion':  conclusion,
        })

    # ── H0-1: CIF — pairwise vs each other model ─────────────────────────
    if has_cif:
        for other in ['InceptionTime','Random Forest','Logistic Regression']:
            if other not in models_here: continue

            # Accuracy
            res = pw_acc_result(exp, 'CIF', other)
            if res:
                stat, p, sig, winner, h, mag = res
                cif_higher = (winner == 'CIF')
                direction  = 'CIF outperforms' if cif_higher else f'{other} outperforms CIF'
                conclusion = (
                    f'{direction} significantly in accuracy (McNemar χ²={stat:.2f}, p={p:.4f}, '
                    f"Cohen's h={h:.3f} [{mag}]). "
                    + ('H1-1 accuracy prediction not supported — CIF does not achieve higher accuracy.' if not cif_higher
                       else 'Consistent with H1-1 accuracy prediction.')
                ) if sig else (
                    f'No significant accuracy difference between CIF and {other} '
                    f'(McNemar χ²={stat:.2f}, p={p:.4f}). H0-1 retained for this pair.'
                )
                decision_rows.append({
                    'Experiment':  exp,
                    'Hypothesis':  f'H0-1 (CIF vs {other})',
                    'Dimension':   'Accuracy',
                    'Test':        "McNemar's (mid-p) + Bonferroni",
                    'Statistic':   f'χ² = {stat:.3f}',
                    'p-value':     f'{p:.4f}',
                    'α threshold': f'{ALPHA_BONF:.4f}',
                    'Decision':    decision_str(sig),
                    'Conclusion':  conclusion,
                })

            # Latency
            res = pw_lat_result(exp, 'CIF', other)
            if res:
                stat, p, sig, faster, delta, mag = res
                cif_faster = (faster == 'CIF')
                conclusion = (
                    f'{"CIF" if cif_faster else other} has significantly lower latency '
                    f"(Mann-Whitney U={stat:.0f}, p={p:.4f}, Cliff's δ={delta:.3f} [{mag}]). "
                    + ('CIF latency claim not supported.' if not cif_faster else 'CIF comparable or faster.')
                ) if sig else (
                    f'No significant latency difference between CIF and {other} '
                    f'(U={stat:.0f}, p={p:.4f}). H0-1 latency component retained.'
                )
                decision_rows.append({
                    'Experiment':  exp,
                    'Hypothesis':  f'H0-1 (CIF vs {other})',
                    'Dimension':   'Latency',
                    'Test':        'Mann-Whitney U + Bonferroni',
                    'Statistic':   f'U = {stat:.0f}',
                    'p-value':     f'{p:.4f}',
                    'α threshold': f'{ALPHA_BONF:.4f}',
                    'Decision':    decision_str(sig),
                    'Conclusion':  conclusion,
                })

    # ── H0-2: InceptionTime vs CIF ────────────────────────────────────────
    if 'InceptionTime' in models_here and has_cif:
        for dim, fn, label in [('Accuracy', pw_acc_result, 'McNemar'),
                                ('Latency',  pw_lat_result, 'Mann-Whitney U')]:
            res = fn(exp, 'InceptionTime', 'CIF')
            if res is None: continue
            stat, p, sig, winner_or_faster, effect, mag = res
            if dim == 'Accuracy':
                it_higher = (winner_or_faster == 'InceptionTime')
                stat_str  = f'χ² = {stat:.3f}'
                thresh    = f'{ALPHA_BONF:.4f}'
                conclusion = (
                    f'InceptionTime {"significantly" if sig else "does not significantly"} outperforms CIF in accuracy '
                    f'(McNemar χ²={stat:.2f}, p={p:.4f}, Cohen\'s h={effect:.3f} [{mag}]). '
                    + ('Accuracy prediction of H1-2 supported.' if (sig and it_higher)
                       else 'Accuracy prediction of H1-2 not supported.')
                )
            else:
                it_slower = (winner_or_faster == 'CIF')  # H1-2 predicts IT has greater latency
                stat_str  = f'U = {stat:.0f}'
                thresh    = f'{ALPHA_BONF:.4f}'
                conclusion = (
                    f'{winner_or_faster} has significantly lower latency '
                    f'(U={stat:.0f}, p={p:.4f}, Cliff\'s δ={effect:.3f} [{mag}]). '
                    + ('H1-2 latency prediction supported (IT slower than CIF).' if (sig and it_slower)
                       else 'H1-2 latency prediction not supported — InceptionTime is faster than CIF, '
                            'a stronger result than hypothesised.')
                ) if sig else f'No significant latency difference between InceptionTime and CIF (U={stat:.0f}, p={p:.4f}).'
            decision_rows.append({
                'Experiment':  exp,
                'Hypothesis':  'H0-2 (InceptionTime vs CIF)',
                'Dimension':   dim,
                'Test':        f"{'McNemar (mid-p) + Bonferroni' if dim=='Accuracy' else 'Mann-Whitney U + Bonferroni'}",
                'Statistic':   stat_str,
                'p-value':     f'{p:.4f}',
                'α threshold': thresh,
                'Decision':    decision_str(sig),
                'Conclusion':  conclusion,
            })

    # ── H0-3: Random Forest ───────────────────────────────────────────────
    if 'Random Forest' in models_here:
        for other in ['InceptionTime','CIF']:
            if other not in models_here: continue
            for dim, fn in [('Accuracy', pw_acc_result), ('Latency', pw_lat_result)]:
                res = fn(exp, 'Random Forest', other)
                if res is None: continue
                stat, p, sig, winner_or_faster, effect, mag = res
                if dim == 'Accuracy':
                    rf_lower = (winner_or_faster != 'Random Forest')
                    stat_str = f'χ² = {stat:.3f}'
                    thresh   = f'{ALPHA_BONF:.4f}'
                    conclusion = (
                        f'Random Forest {"has significantly lower" if (sig and rf_lower) else "does not have significantly lower"} '
                        f'accuracy than {other} (McNemar χ²={stat:.2f}, p={p:.4f}, '
                        f"Cohen's h={effect:.3f} [{mag}]). "
                        + (f'H1-3 accuracy prediction vs {other} {"supported" if rf_lower else "not supported"} '
                           f'({"RF < " if rf_lower else "RF ≥ "}{other}).')
                    )
                else:
                    rf_faster = (winner_or_faster == 'Random Forest')
                    stat_str  = f'U = {stat:.0f}'
                    thresh    = f'{ALPHA_BONF:.4f}'
                    conclusion = (
                        f'Random Forest {"is significantly" if sig else "is not significantly"} '
                        f'{"faster" if rf_faster else "slower"} than {other} '
                        f"(U={stat:.0f}, p={p:.4f}, Cliff's δ={effect:.3f} [{mag}]). "
                        + (f'H1-3 efficiency prediction vs {other} {"supported" if rf_faster else "not supported"}.')
                    )
                decision_rows.append({
                    'Experiment':  exp,
                    'Hypothesis':  f'H0-3 (RF vs {other})',
                    'Dimension':   dim,
                    'Test':        f"{'McNemar (mid-p) + Bonferroni' if dim=='Accuracy' else 'Mann-Whitney U + Bonferroni'}",
                    'Statistic':   stat_str,
                    'p-value':     f'{p:.4f}',
                    'α threshold': thresh,
                    'Decision':    decision_str(sig),
                    'Conclusion':  conclusion,
                })

    # ── H0-4: Logistic Regression ─────────────────────────────────────────
    if 'Logistic Regression' in models_here:
        for other in ['InceptionTime','Random Forest','CIF']:
            if other not in models_here: continue
            for dim, fn in [('Accuracy', pw_acc_result), ('Latency', pw_lat_result)]:
                res = fn(exp, 'Logistic Regression', other)
                if res is None: continue
                stat, p, sig, winner_or_faster, effect, mag = res
                if dim == 'Accuracy':
                    lr_lower = (winner_or_faster != 'Logistic Regression')
                    stat_str = f'χ² = {stat:.3f}'
                    thresh   = f'{ALPHA_BONF:.4f}'
                    conclusion = (
                        f'Logistic Regression {"has significantly lower" if (sig and lr_lower) else "does not have significantly lower"} '
                        f'accuracy than {other} (McNemar χ²={stat:.2f}, p={p:.4f}, '
                        f"Cohen's h={effect:.3f} [{mag}]). "
                        + (f'H1-4 lowest-accuracy prediction vs {other} {"supported" if lr_lower else "not supported"}.')
                    )
                else:
                    lr_faster = (winner_or_faster == 'Logistic Regression')
                    stat_str  = f'U = {stat:.0f}'
                    thresh    = f'{ALPHA_BONF:.4f}'
                    conclusion = (
                        f'Logistic Regression {"is significantly" if sig else "is not significantly"} '
                        f'{"faster" if lr_faster else "slower"} than {other} '
                        f"(U={stat:.0f}, p={p:.4f}, Cliff's δ={effect:.3f} [{mag}]). "
                        + (f'H1-4 highest-efficiency prediction vs {other} {"supported" if lr_faster else "not supported"}.')
                    )
                decision_rows.append({
                    'Experiment':  exp,
                    'Hypothesis':  f'H0-4 (LR vs {other})',
                    'Dimension':   dim,
                    'Test':        f"{'McNemar (mid-p) + Bonferroni' if dim=='Accuracy' else 'Mann-Whitney U + Bonferroni'}",
                    'Statistic':   stat_str,
                    'p-value':     f'{p:.4f}',
                    'α threshold': thresh,
                    'Decision':    decision_str(sig),
                    'Conclusion':  conclusion,
                })

decision_df = pd.DataFrame(decision_rows)

print('Per-RQ Formal Decision Table')
print('=' * 100)
display(decision_df)

decision_df.to_csv(os.path.join(STAT_DIR, 'per_rq_decision_table_154class.csv'), index=False)
print(f'\n\u2705 Saved \u2192 results/statistical/per_rq_decision_table_154class.csv')

In [ ]:
# ── Cell 16: Per-model synthesis table ──────────────────────────────────
# One row per model — consolidates accuracy and latency verdicts
# across all experiments into a single readable summary.
# This is the table most directly usable in a dissertation results chapter.

# Aggregate McNemar wins/losses for each model
def model_acc_summary(model):
    """Return (wins, losses, ties, avg_h) where wins = pairs where model sig. outperforms."""
    subset = mcnemar_df[
        ((mcnemar_df['Model A'] == model) | (mcnemar_df['Model B'] == model))
    ]
    wins   = subset[(subset['Higher acc'] == model) & (subset[BONF_COL] == 'YES')]
    losses = subset[(subset['Higher acc'] != model) & (subset[BONF_COL] == 'YES')]
    ties   = subset[subset[BONF_COL] == 'NO']
    avg_h  = subset["Cohen's h"].abs().mean()
    return len(wins), len(losses), len(ties), avg_h

def model_lat_summary(model):
    """Return (faster_wins, slower_wins, ties, avg_delta)."""
    subset = mwu_df[
        ((mwu_df['Model A'] == model) | (mwu_df['Model B'] == model))
    ]
    faster_wins  = subset[(subset['Faster model'] == model)  & (subset[BONF_COL] == 'YES')]
    slower_wins  = subset[(subset['Faster model'] != model)  & (subset[BONF_COL] == 'YES')]
    ties         = subset[subset[BONF_COL] == 'NO']
    avg_delta    = subset["Cliff's delta"].abs().mean()
    return len(faster_wins), len(slower_wins), len(ties), avg_delta

def overall_acc_rank(model, all_models=MODELS_ORDER):
    """Rank model by mean accuracy across all experiments."""
    means = {}
    for m in all_models:
        vals = [accuracy_score(*preds[e][m]) for e in preds if m in preds[e]]
        means[m] = np.mean(vals) if vals else 0
    ranked = sorted(means, key=means.get, reverse=True)
    return ranked.index(model) + 1 if model in ranked else '—'

LATENCY_RANK_ORDER = ['Logistic Regression','InceptionTime','Random Forest','CIF']

model_summary_rows = []

for model in MODELS_ORDER:
    # Accuracy
    acc_vals = [accuracy_score(*preds[e][model]) * 100 for e in preds if model in preds.get(e, {})]
    mean_acc = np.mean(acc_vals) if acc_vals else None
    n_exps   = len(acc_vals)
    acc_wins, acc_losses, acc_ties, avg_h = model_acc_summary(model)
    lat_wins, lat_losses, lat_ties, avg_d  = model_lat_summary(model)
    acc_rank = overall_acc_rank(model)
    lat_rank = LATENCY_RANK_ORDER.index(model) + 1 if model in LATENCY_RANK_ORDER else '—'

    # Hypothesis verdict
    hyp_map = {
        'CIF':                'H1-1: CIF not supported — lowest accuracy, highest latency',
        'InceptionTime':      'H1-2: Partially supported — highest accuracy confirmed; IT also faster than CIF (stronger than hypothesised)',
        'Random Forest':      'H1-3: Partially supported — RF < IT accuracy (confirmed); RF > CIF accuracy (contradicted)',
        'Logistic Regression':'H1-4: Partially supported — near-highest efficiency confirmed; not lowest accuracy (CIF is lower)',
    }

    # Overall decision across experiments
    h0_decisions = decision_df[
        decision_df['Hypothesis'].str.contains(model[:3] if model != 'CIF' else 'CIF')
    ]['Decision'].value_counts()
    majority_decision = h0_decisions.idxmax() if len(h0_decisions) else '—'

    model_summary_rows.append({
        'Model':                   model,
        'Experiments run':         n_exps,
        'Mean accuracy (%)':       f'{mean_acc:.2f}' if mean_acc else '—',
        'Accuracy rank':           acc_rank,
        'Latency rank (fastest=1)': lat_rank,
        'Acc pairwise wins':       acc_wins,
        'Acc pairwise losses':     acc_losses,
        'Acc non-sig pairs':       acc_ties,
        'Avg Cohen h':             f'{avg_h:.3f}',
        'Lat pairwise wins':       lat_wins,
        'Lat pairwise losses':     lat_losses,
        'Avg Cliff delta':         f'{avg_d:.3f}',
        'Hypothesis':              'H1-' + ('1' if model=='CIF' else '2' if model=='InceptionTime'
                                            else '3' if model=='Random Forest' else '4'),
        'Overall verdict':         hyp_map.get(model, '—'),
    })

model_summary_df = pd.DataFrame(model_summary_rows)

print('Per-Model Synthesis Table')
print('=' * 100)
display(model_summary_df.set_index('Model'))

model_summary_df.to_csv(os.path.join(STAT_DIR, 'per_model_synthesis_table_154class.csv'), index=False)
print(f'\n\u2705 Saved \u2192 results/statistical/per_model_synthesis_table_154class.csv')

In [ ]:
# ── Cell 17: Master dissertation results table ───────────────────────────
# A single consolidated table per experiment × model showing:
# Model | Acc | F1 | Acc rank | Global test (Q/H) | Best pairwise p | Decision | Conclusion
# Format is designed to be copy-pasted directly into a LaTeX or Word table.

master_rows = []

for exp in ['154-class']:
    if exp not in preds: continue

    models_here = [m for m in MODELS_ORDER if m in preds[exp]]

    # Global test results
    q_res  = global_acc_result(exp)
    kw_res = global_lat_result(exp)

    for model in models_here:
        y_true, y_pred = preds[exp][model]
        acc = accuracy_score(y_true, y_pred) * 100
        f1  = f1_score(y_true, y_pred, average='macro', zero_division=0) * 100

        # CI for this model×experiment
        ci_row = ci_df[(ci_df['Experiment'] == exp) & (ci_df['Model'] == model)]
        if len(ci_row):
            f1_lo  = ci_row.iloc[0]['F1 CI low']  * 100
            f1_hi  = ci_row.iloc[0]['F1 CI high'] * 100
            ci_str = f'{f1:.2f}% [{f1_lo:.2f}, {f1_hi:.2f}]'
        else:
            ci_str = f'{f1:.2f}%'

        # Best (lowest) pairwise p-value involving this model
        pw_rows = mcnemar_df[
            ((mcnemar_df['Model A'] == model) | (mcnemar_df['Model B'] == model)) &
            (mcnemar_df['Experiment'] == exp)
        ]
        min_p = pw_rows['p-value'].min() if len(pw_rows) else None
        n_sig_pairs = (pw_rows[BONF_COL] == 'YES').sum() if BONF_COL in pw_rows.columns else 0

        # Hypothesis label
        hyp = {'InceptionTime': 'H1-2', 'Random Forest': 'H1-3',
               'Logistic Regression': 'H1-4', 'CIF': 'H1-1'}.get(model, '—')

        # Acc global significance
        global_acc_sig = 'Sig.' if (q_res and q_res[2]) else 'n.s.'
        global_lat_sig = 'Sig.' if (kw_res and kw_res[2]) else 'n.s.'

        # Written conclusion (brief — for table footnote style)
        hyp_conclusions = {
            ('InceptionTime', 'Accuracy'):      'Highest accuracy; H1-2 accuracy prediction confirmed.',
            ('InceptionTime', 'Latency'):       'Fastest of deep/tree models; H1-2 latency prediction reversed.',
            ('Random Forest',  'Accuracy'):     'Second/mid accuracy; H1-3 partially confirmed.',
            ('Random Forest',  'Latency'):      'Faster than CIF, slower than IT/LR; H1-3 efficiency partially confirmed.',
            ('Logistic Regression','Accuracy'): 'Near-lowest accuracy (above CIF); H1-4 lowest-acc prediction not confirmed.',
            ('Logistic Regression','Latency'):  'Near-highest efficiency (tied with IT); H1-4 efficiency prediction confirmed.',
            ('CIF', 'Accuracy'):                'Lowest accuracy across all experiments; H1-1 rejected.',
            ('CIF', 'Latency'):                 'Highest latency (258-697x slower than others); H1-1 latency claim rejected.',
        }
        acc_note = hyp_conclusions.get((model, 'Accuracy'), '—')
        lat_note = hyp_conclusions.get((model, 'Latency'), '—')

        master_rows.append({
            'Experiment':              exp,
            'Model':                   model,
            'Hypothesis':              hyp,
            'Accuracy (%)':            f'{acc:.2f}',
            'Macro F1 (%) [95% CI]':   ci_str,
            'Global acc test':         f"Cochran Q {'*' if q_res and q_res[2] else 'n.s.'}",
            'Global lat test':         f"Kruskal H {'*' if kw_res and kw_res[2] else 'n.s.'}",
            'Min pairwise p (acc)':    f'{min_p:.4f}' if min_p is not None else '—',
            'Sig. pairwise pairs':     n_sig_pairs,
            'Accuracy conclusion':     acc_note,
            'Latency conclusion':      lat_note,
        })

master_df = pd.DataFrame(master_rows)

print('Master Dissertation Results Table')
print('=' * 110)
for exp_grp, grp in master_df.groupby('Experiment', sort=False):
    print(f'\n── {exp_grp} ──')
    display(grp.drop(columns='Experiment').set_index('Model'))

master_df.to_csv(os.path.join(STAT_DIR, 'master_dissertation_table_154class.csv'), index=False)
print(f'\n\u2705 Saved \u2192 results/statistical/master_dissertation_table_154class.csv')

print('\n━━━ Files produced by the decision table cells ━━━')
for f in ['per_rq_decision_table.csv','per_model_synthesis_table.csv','master_dissertation_table.csv']:
    fpath = os.path.join(STAT_DIR, f)
    exists = '✅' if os.path.exists(fpath) else '⚠️'
    print(f'  {exists}  {f}')

In [ ]:
# ── Cell 15: Final summary printout ─────────────────────────────────────

print('=' * 80)
print('  STATISTICAL TESTING COMPLETE')
print('=' * 80)
print(f'\nAll outputs saved to: {STAT_DIR}\n')

output_files = [
    ('bootstrap_confidence_intervals_154class.csv',              '95% CI for Accuracy & F1 per model×experiment'),
    ('global_cochrans_q_accuracy_154class.csv',                  'Cochran\'s Q — global accuracy test (all models simultaneously)'),
    ('global_kruskal_wallis_latency_154class.csv',               'Kruskal-Wallis — global latency test'),
    ('pairwise_mcnemar_accuracy_154class.csv',                   'McNemar\'s pairwise accuracy tests + Cohen\'s h + Bonferroni'),
    ('pairwise_mannwhitney_latency_154class.csv',                'Mann-Whitney U pairwise latency tests + Cliff\'s delta + Bonferroni'),
    ('hypothesis_verdicts_154class.csv',                         'Verdict per RQ hypothesis per experiment (programmatic)'),
    ('per_rq_decision_table_154class.csv',                       'Formal decision table: test → statistic → p → decision → conclusion'),
    ('per_model_synthesis_table_154class.csv',                   'Per-model synthesis: wins/losses, effect sizes, overall verdict'),
    ('master_dissertation_table_154class.csv',                   'Master table: all models × experiments with full statistical conclusions'),
    ('effect_size_summary_154class.png',                         'Forest plot of Cohen\'s h and Cliff\'s delta (averaged)'),
    ('ci_forest_plot_f1_154class.png',                           'CI forest plot — Macro F1 per model×experiment'),
    ('mcnemar_heatmap_154class_*.png',                           'p-value heatmaps per experiment'),
]

print(f'  {"File":<55} Description')
print(f'  {"-"*54} {"-"*35}')
for fname, desc in output_files:
    fpath = os.path.join(STAT_DIR, fname)
    exists = '✅' if (os.path.exists(fpath) or '*' in fname) else '⚠️ '
    print(f'  {exists}  {fname:<53} {desc}')

print('\nAlpha levels used:')
print(f'  Global tests (Cochran Q, Kruskal-Wallis) : α = {ALPHA}')
print(f'  Pairwise tests with Bonferroni correction : α = {ALPHA_BONF:.4f}  ({N_PAIRS} pairs)')
print(f'  Bootstrap CI reps : {N_BOOTSTRAP}')
print(f'  Latency trials    : {N_LATENCY_TRIALS}')

---
## Section 16 — Per-Model Hypothesis Conclusion Tables

Each table below answers one research question with a structured view of:
- The specific sub-hypothesis being tested
- Which test was used and why
- The test statistic and p-value (averaged across experiments where applicable)
- The Bonferroni-corrected decision threshold
- A formal **Reject / Fail to Reject** decision
- Effect size and plain-English interpretation

These tables are dissertation-ready and saved as CSV and styled HTML.

> **Note on ANOVA:** One-way ANOVA was considered but is **not appropriate** here.  
> Accuracy metrics are bounded proportions (not normally distributed), and latency  
> measurements are right-skewed. The non-parametric equivalents used throughout  
> (Cochran's Q, Kruskal-Wallis, McNemar's, Mann-Whitney U) make no distributional  
> assumptions and are the correct choice for this analysis.

In [ ]:
# ── Cell 16B: Global hypothesis table (H0G) ──────────────────────────────

global_rows = []

# Cochran's Q — accuracy
q_mean_stat = cochran_df['Cochran Q'].mean()
q_mean_p    = cochran_df['p-value'].mean()
q_any_sig   = (cochran_df['Significant (α=0.05)'] == 'YES').any()

global_rows.append({
    'Hypothesis':    'H0G',
    'Sub-hypothesis': 'No significant accuracy differences across all 4 models',
    'Dimension':     'Accuracy',
    'Test':          "Cochran's Q",
    'Rationale':     'Non-parametric; compares ≥3 classifiers on same test set',
    'Statistic (avg)': f'Q = {q_mean_stat:.3f}',
    'p-value (avg)': f'{q_mean_p:.4f}',
    'α threshold':   f'{ALPHA} (global)',
    'Decision':      'Reject H₀' if q_any_sig else 'Fail to Reject H₀',
    'Effect size':   '—',
    'Interpretation': ('Significant accuracy differences exist across models'
                       if q_any_sig else
                       'No significant global accuracy difference detected'),
})

# Kruskal-Wallis — latency
kw_mean_stat = kw_df['Kruskal H'].mean()
kw_mean_p    = kw_df['p-value'].mean()
kw_any_sig   = (kw_df['Significant (α=0.05)'] == 'YES').any()

global_rows.append({
    'Hypothesis':    'H0G',
    'Sub-hypothesis': 'No significant latency differences across all 4 models',
    'Dimension':     'Latency',
    'Test':          'Kruskal-Wallis',
    'Rationale':     'Non-parametric ANOVA equivalent; latency is right-skewed',
    'Statistic (avg)': f'H = {kw_mean_stat:.3f}',
    'p-value (avg)': f'{kw_mean_p:.4f}',
    'α threshold':   f'{ALPHA} (global)',
    'Decision':      'Reject H₀' if kw_any_sig else 'Fail to Reject H₀',
    'Effect size':   '—',
    'Interpretation': ('Significant latency differences exist across models'
                       if kw_any_sig else
                       'No significant global latency difference detected'),
})

global_df = pd.DataFrame(global_rows)
print('H0G — Global Hypothesis Conclusion Table')
print('=' * 90)
display(global_df.style
    .set_caption('Table: Global Hypothesis H0G — Statistical Conclusions')
    .applymap(lambda v: 'color: #C62828; font-weight: bold' if 'Reject H₀' == v and 'Fail' not in v else
              ('color: #1565C0; font-weight: bold' if 'Fail' in str(v) else ''),
              subset=['Decision'])
    .set_properties(**{'font-size': '12px', 'text-align': 'left'})
    .hide(axis='index')
)

global_df.to_csv(os.path.join(STAT_DIR, 'table_H0G_global_hypothesis_154class.csv'), index=False)
print(f'\n✅ Saved → results/statistical/table_H0G_global_hypothesis_154class.csv')

In [ ]:
# ── Cell 16C: CIF conclusion table (H1-1) ───────────────────────────────
# H1-1: CIF achieves significantly higher accuracy and comparable/lower latency
# than all other models.

FOCAL = 'CIF'
OTHERS = ['InceptionTime', 'Random Forest', 'Logistic Regression']

cif_rows = []

for exp in sorted(preds.keys()):
    if FOCAL not in preds.get(exp, {}):
        continue

    # --- Accuracy tests vs each other model ---
    for other in OTHERS:
        if other not in preds.get(exp, {}):
            continue
        r = get_sig_pairs(mcnemar_df, exp, FOCAL, other)
        if r is None:
            continue
        bonf_col = [c for c in r.index if 'Bonf' in c][0]
        focal_acc = r['Acc A'] if r['Model A'] == FOCAL else r['Acc B']
        other_acc = r['Acc B'] if r['Model A'] == FOCAL else r['Acc A']
        direction = 'CIF > ' + other if focal_acc > other_acc else 'CIF < ' + other
        cohenh_val = r["Cohen's h"]

        cif_rows.append({
            'Experiment':        exp,
            'Hypothesis':        'H1-1',
            'Sub-hypothesis':    f'CIF accuracy > {other}',
            'Dimension':         'Accuracy',
            'Test':              "McNemar's (mid-p)",
            'Statistic':         f"{r['McNemar stat']:.3f}",
            'p-value':           r['p-value'],
            'α (Bonferroni)':    f'{ALPHA_BONF:.4f}',
            'Decision':          'Reject H₀' if r[bonf_col] == 'YES' else 'Fail to Reject H₀',
            'Observed direction': direction,
            "Cohen's h":         f'{cohenh_val:.3f}',
            'Effect size':       r['Effect size'],
            'H1-1 supported?':   ('NO — CIF not significantly better'
                                  if (r[bonf_col] == 'YES' and focal_acc < other_acc) or r[bonf_col] == 'NO'
                                  else 'YES'),
        })

    # --- Latency test vs each other model ---
    for other in OTHERS:
        if other not in latency.get(exp, {}):
            continue
        r_lat = get_sig_pairs(mwu_df, exp, FOCAL, other)
        if r_lat is None:
            continue
        bonf_col = [c for c in r_lat.index if 'Bonf' in c][0]
        focal_med = r_lat['Median A (ms)'] if r_lat['Model A'] == FOCAL else r_lat['Median B (ms)']
        other_med = r_lat['Median B (ms)'] if r_lat['Model A'] == FOCAL else r_lat['Median A (ms)']
        direction = f'CIF ({focal_med:.1f}ms) vs {other} ({other_med:.1f}ms)'
        cliffsd_val = r_lat["Cliff's delta"]

        cif_rows.append({
            'Experiment':        exp,
            'Hypothesis':        'H1-1',
            'Sub-hypothesis':    f'CIF latency comparable to {other}',
            'Dimension':         'Latency',
            'Test':              'Mann-Whitney U',
            'Statistic':         f"{r_lat['MWU statistic']:.1f}",
            'p-value':           r_lat['p-value'],
            'α (Bonferroni)':    f'{ALPHA_BONF:.4f}',
            'Decision':          'Reject H₀' if r_lat[bonf_col] == 'YES' else 'Fail to Reject H₀',
            'Observed direction': direction,
            "Cohen's h":         f"{cliffsd_val:.3f} (Cliff's δ)",
            'Effect size':       r_lat['Effect size'],
            'H1-1 supported?':   ('NO — CIF significantly slower'
                                  if r_lat[bonf_col] == 'YES' and focal_med > other_med
                                  else 'Comparable'),
        })

cif_table = pd.DataFrame(cif_rows)
print('H1-1 — CIF Hypothesis Conclusion Table')
print('=' * 90)
display(
    cif_table.style
    .set_caption('Table H1-1: CIF — Statistical Conclusions by Experiment & Comparison')
    .applymap(lambda v: 'color: #C62828; font-weight: bold' if v == 'Reject H₀' else
              ('color: #1565C0; font-weight: bold' if 'Fail' in str(v) else ''),
              subset=['Decision'])
    .applymap(lambda v: 'color: #C62828' if 'NO' in str(v) else
              ('color: #2E7D32' if 'YES' in str(v) else ''),
              subset=['H1-1 supported?'])
    .set_properties(**{'font-size': '11px', 'text-align': 'left'})
    .hide(axis='index')
)

cif_table.to_csv(os.path.join(STAT_DIR, 'table_H1-1_CIF_conclusions_154class.csv'), index=False)
print(f'\n✅ Saved → results/statistical/table_H1-1_CIF_conclusions_154class.csv')

In [ ]:
# ── Cell 16D: InceptionTime conclusion table (H1-2) ─────────────────────
# H1-2: InceptionTime achieves significantly higher accuracy but greater
# latency than CIF.

FOCAL = 'InceptionTime'

it_rows = []

for exp in sorted(preds.keys()):
    if FOCAL not in preds.get(exp, {}):
        continue

    for other, sub_hyp, lat_direction_expected in [
        ('CIF',                 'InceptionTime accuracy > CIF',           'IT slower (H1-2 predicts)'),
        ('Random Forest',       'InceptionTime accuracy > Random Forest', 'N/A'),
        ('Logistic Regression', 'InceptionTime accuracy > LR',            'N/A'),
    ]:
        if other not in preds.get(exp, {}):
            continue

        # Accuracy
        r = get_sig_pairs(mcnemar_df, exp, FOCAL, other)
        if r is None:
            continue
        bonf_col  = [c for c in r.index if 'Bonf' in c][0]
        focal_acc = r['Acc A'] if r['Model A'] == FOCAL else r['Acc B']
        other_acc = r['Acc B'] if r['Model A'] == FOCAL else r['Acc A']
        direction = f'IT ({focal_acc*100:.1f}%) vs {other} ({other_acc*100:.1f}%)'
        h12_acc   = ('YES — IT significantly better' if r[bonf_col] == 'YES' and focal_acc > other_acc
                     else 'NO')
        cohenh_val = r["Cohen's h"]

        it_rows.append({
            'Experiment':        exp,
            'Hypothesis':        'H1-2',
            'Sub-hypothesis':    sub_hyp,
            'Dimension':         'Accuracy',
            'Test':              "McNemar's (mid-p)",
            'Statistic':         f"{r['McNemar stat']:.3f}",
            'p-value':           r['p-value'],
            'α (Bonferroni)':    f'{ALPHA_BONF:.4f}',
            'Decision':          'Reject H₀' if r[bonf_col] == 'YES' else 'Fail to Reject H₀',
            'Observed direction': direction,
            "Cohen's h":         f'{cohenh_val:.3f}',
            'Effect size':       r['Effect size'],
            'H1-2 supported?':   h12_acc,
        })

        # Latency (only meaningful for CIF comparison per H1-2)
        if other == 'CIF':
            r_lat = get_sig_pairs(mwu_df, exp, FOCAL, other)
            if r_lat is not None:
                bonf_lat  = [c for c in r_lat.index if 'Bonf' in c][0]
                focal_med = r_lat['Median A (ms)'] if r_lat['Model A'] == FOCAL else r_lat['Median B (ms)']
                other_med = r_lat['Median B (ms)'] if r_lat['Model A'] == FOCAL else r_lat['Median A (ms)']
                it_slower = focal_med > other_med
                h12_lat   = ('CONTRADICTED — IT actually faster than CIF'
                              if not it_slower else 'Confirmed — IT slower than CIF')
                cliffsd_val = r_lat["Cliff's delta"]

                it_rows.append({
                    'Experiment':        exp,
                    'Hypothesis':        'H1-2',
                    'Sub-hypothesis':    'InceptionTime latency > CIF (H1-2 predicts IT slower)',
                    'Dimension':         'Latency',
                    'Test':              'Mann-Whitney U',
                    'Statistic':         f"{r_lat['MWU statistic']:.1f}",
                    'p-value':           r_lat['p-value'],
                    'α (Bonferroni)':    f'{ALPHA_BONF:.4f}',
                    'Decision':          'Reject H₀' if r_lat[bonf_lat] == 'YES' else 'Fail to Reject H₀',
                    'Observed direction': f'IT ({focal_med:.1f}ms) vs CIF ({other_med:.1f}ms)',
                    "Cohen's h":         f"{cliffsd_val:.3f} (Cliff's δ)",
                    'Effect size':       r_lat['Effect size'],
                    'H1-2 supported?':   h12_lat,
                })

it_table = pd.DataFrame(it_rows)
print('H1-2 — InceptionTime Hypothesis Conclusion Table')
print('=' * 90)
display(
    it_table.style
    .set_caption('Table H1-2: InceptionTime — Statistical Conclusions by Experiment & Comparison')
    .applymap(lambda v: 'color: #C62828; font-weight: bold' if v == 'Reject H₀' else
              ('color: #1565C0; font-weight: bold' if 'Fail' in str(v) else ''),
              subset=['Decision'])
    .applymap(lambda v: 'color: #C62828' if 'CONTRADICTED' in str(v) or v == 'NO' else
              ('color: #2E7D32' if 'YES' in str(v) or 'Confirmed' in str(v) else ''),
              subset=['H1-2 supported?'])
    .set_properties(**{'font-size': '11px', 'text-align': 'left'})
    .hide(axis='index')
)

it_table.to_csv(os.path.join(STAT_DIR, 'table_H1-2_InceptionTime_conclusions_154class.csv'), index=False)
print(f'\n✅ Saved → results/statistical/table_H1-2_InceptionTime_conclusions_154class.csv')

In [ ]:
# ── Cell 16E: Random Forest conclusion table (H1-3) ──────────────────────
# H1-3: RF achieves lower accuracy than InceptionTime but greater
# computational efficiency than CIF or InceptionTime.
FOCAL = 'Random Forest'

rf_rows = []

for exp in sorted(preds.keys()):
    if FOCAL not in preds.get(exp, {}): continue

    for other, sub_acc, acc_direction_expected, sub_lat, lat_direction_expected in [
        ('InceptionTime',
         'RF accuracy < InceptionTime', 'RF lower (H1-3 predicts)',
         'RF latency < InceptionTime',  'RF faster (H1-3 predicts)'),
        ('CIF',
         'RF accuracy > CIF',           'RF higher than CIF (contradicts H1-3)',
         'RF latency < CIF',            'RF faster (H1-3 predicts)'),
        ('Logistic Regression',
         'RF vs LR accuracy',           'Informational',
         'RF vs LR latency',            'Informational'),
    ]:
        if other not in preds.get(exp, {}): continue

        # Accuracy
        r = get_sig_pairs(mcnemar_df, exp, FOCAL, other)
        if r is not None:
            bonf_col  = [c for c in r.index if 'Bonf' in c][0]
            focal_acc = r['Acc A'] if r['Model A'] == FOCAL else r['Acc B']
            other_acc = r['Acc B'] if r['Model A'] == FOCAL else r['Acc A']
            sig       = r[bonf_col] == 'YES'
            cohenh_val = r["Cohen's h"]

            if other == 'InceptionTime':
                supported = ('YES — RF significantly lower' if sig and focal_acc < other_acc
                             else 'NO — not significantly different' if not sig
                             else 'NO — RF higher than IT')
            elif other == 'CIF':
                supported = ('UNEXPECTED — RF significantly better than CIF' if sig and focal_acc > other_acc
                             else 'As expected — RF better than CIF')
            else:
                supported = f'RF {"higher" if focal_acc > other_acc else "lower"} (informational)'

            rf_rows.append({
                'Experiment':        exp,
                'Hypothesis':        'H1-3',
                'Sub-hypothesis':    sub_acc,
                'Dimension':         'Accuracy',
                'Test':              "McNemar's (mid-p)",
                'Statistic':         f"{r['McNemar stat']:.3f}",
                'p-value':           r['p-value'],
                'α (Bonferroni)':    f'{ALPHA_BONF:.4f}',
                'Decision':          'Reject H₀' if sig else 'Fail to Reject H₀',
                'Observed':          f'RF ({focal_acc*100:.1f}%) vs {other} ({other_acc*100:.1f}%)',
                "Cohen's h":         f'{cohenh_val:.3f}',
                'Effect size':       r['Effect size'],
                'H1-3 supported?':   supported,
            })

        # Latency
        r_lat = get_sig_pairs(mwu_df, exp, FOCAL, other)
        if r_lat is not None and other in latency.get(exp, {}):
            bonf_lat  = [c for c in r_lat.index if 'Bonf' in c][0]
            focal_med = r_lat['Median A (ms)'] if r_lat['Model A'] == FOCAL else r_lat['Median B (ms)']
            other_med = r_lat['Median B (ms)'] if r_lat['Model A'] == FOCAL else r_lat['Median A (ms)']
            sig_lat   = r_lat[bonf_lat] == 'YES'
            rf_faster = focal_med < other_med
            cliffsd_val = r_lat["Cliff's delta"]

            if other in ('InceptionTime', 'CIF'):
                supported_lat = ('YES — RF significantly faster' if sig_lat and rf_faster
                                 else 'NO — RF not faster' if not rf_faster
                                 else 'Not significant')
            else:
                supported_lat = f'RF {"faster" if rf_faster else "slower"} than {other} (informational)'

            rf_rows.append({
                'Experiment':        exp,
                'Hypothesis':        'H1-3',
                'Sub-hypothesis':    sub_lat,
                'Dimension':         'Latency',
                'Test':              'Mann-Whitney U',
                'Statistic':         f"{r_lat['MWU statistic']:.1f}",
                'p-value':           r_lat['p-value'],
                'α (Bonferroni)':    f'{ALPHA_BONF:.4f}',
                'Decision':          'Reject H₀' if sig_lat else 'Fail to Reject H₀',
                'Observed':          f'RF ({focal_med:.1f}ms) vs {other} ({other_med:.1f}ms)',
                "Cohen's h":         f"{cliffsd_val:.3f} (Cliff's δ)",
                'Effect size':       r_lat['Effect size'],
                'H1-3 supported?':   supported_lat,
            })

rf_table = pd.DataFrame(rf_rows)
print('H1-3 — Random Forest Hypothesis Conclusion Table')
print('=' * 90)
display(
    rf_table.style
    .set_caption('Table H1-3: Random Forest — Statistical Conclusions by Experiment & Comparison')
    .applymap(lambda v: 'color: #C62828; font-weight: bold' if v == 'Reject H₀' else
              ('color: #1565C0; font-weight: bold' if 'Fail' in str(v) else ''),
              subset=['Decision'])
    .applymap(lambda v: 'color: #C62828' if v == 'NO — not significantly different' or v == 'NO' else
              ('color: #1565C0' if 'UNEXPECTED' in str(v) else
               ('color: #2E7D32' if 'YES' in str(v) else '')),
              subset=['H1-3 supported?'])
    .set_properties(**{'font-size': '11px', 'text-align': 'left'})
    .hide(axis='index')
)

rf_table.to_csv(os.path.join(STAT_DIR, 'table_H1-3_RF_conclusions_154class.csv'), index=False)
print(f'\n✅ Saved → results/statistical/table_H1-3_RF_conclusions_154class.csv')

In [ ]:
# ── Cell 16F: Logistic Regression conclusion table (H1-4) ────────────────
# H1-4: LR achieves lowest accuracy but highest computational efficiency.
FOCAL = 'Logistic Regression'

lr_rows = []

for exp in sorted(preds.keys()):
    if FOCAL not in preds.get(exp, {}): continue

    for other in ['InceptionTime', 'Random Forest', 'CIF']:
        if other not in preds.get(exp, {}): continue

        # Accuracy
        r = get_sig_pairs(mcnemar_df, exp, FOCAL, other)
        if r is not None:
            bonf_col  = [c for c in r.index if 'Bonf' in c][0]
            focal_acc = r['Acc A'] if r['Model A'] == FOCAL else r['Acc B']
            other_acc = r['Acc B'] if r['Model A'] == FOCAL else r['Acc A']
            sig       = r[bonf_col] == 'YES'
            lr_lower  = focal_acc < other_acc
            cohenh_val = r["Cohen's h"]

            if other == 'InceptionTime':
                supported = ('YES — LR significantly lower than IT' if sig and lr_lower
                             else 'Partially — not significant' if not sig and lr_lower
                             else 'NO — LR unexpectedly higher')
            elif other == 'CIF':
                supported = ('CONTRADICTED — LR better than CIF' if focal_acc > other_acc
                             else 'Consistent — LR lower')
            else:
                supported = f'LR {"lower" if lr_lower else "higher"} than {other}'

            lr_rows.append({
                'Experiment':        exp,
                'Hypothesis':        'H1-4',
                'Sub-hypothesis':    f'LR accuracy lowest vs {other}',
                'Dimension':         'Accuracy',
                'Test':              "McNemar's (mid-p)",
                'Statistic':         f"{r['McNemar stat']:.3f}",
                'p-value':           r['p-value'],
                'α (Bonferroni)':    f'{ALPHA_BONF:.4f}',
                'Decision':          'Reject H₀' if sig else 'Fail to Reject H₀',
                'Observed':          f'LR ({focal_acc*100:.1f}%) vs {other} ({other_acc*100:.1f}%)',
                "Cohen's h":         f'{cohenh_val:.3f}',
                'Effect size':       r['Effect size'],
                'H1-4 supported?':   supported,
            })

        # Latency
        r_lat = get_sig_pairs(mwu_df, exp, FOCAL, other)
        if r_lat is not None and other in latency.get(exp, {}):
            bonf_lat  = [c for c in r_lat.index if 'Bonf' in c][0]
            focal_med = r_lat['Median A (ms)'] if r_lat['Model A'] == FOCAL else r_lat['Median B (ms)']
            other_med = r_lat['Median B (ms)'] if r_lat['Model A'] == FOCAL else r_lat['Median A (ms)']
            sig_lat   = r_lat[bonf_lat] == 'YES'
            lr_faster = focal_med < other_med
            cliffsd_val = r_lat["Cliff's delta"]

            if other == 'InceptionTime':
                supported_lat = ('Effectively tied — no significant difference' if not sig_lat
                                 else 'LR significantly faster than IT' if lr_faster
                                 else 'IT significantly faster than LR')
            else:
                supported_lat = ('YES — LR significantly faster' if sig_lat and lr_faster
                                 else 'NO — not significantly faster' if not sig_lat
                                 else 'NO — LR slower')

            lr_rows.append({
                'Experiment':        exp,
                'Hypothesis':        'H1-4',
                'Sub-hypothesis':    f'LR latency lowest (fastest) vs {other}',
                'Dimension':         'Latency',
                'Test':              'Mann-Whitney U',
                'Statistic':         f"{r_lat['MWU statistic']:.1f}",
                'p-value':           r_lat['p-value'],
                'α (Bonferroni)':    f'{ALPHA_BONF:.4f}',
                'Decision':          'Reject H₀' if sig_lat else 'Fail to Reject H₀',
                'Observed':          f'LR ({focal_med:.1f}ms) vs {other} ({other_med:.1f}ms)',
                "Cohen's h":         f"{cliffsd_val:.3f} (Cliff's δ)",
                'Effect size':       r_lat['Effect size'],
                'H1-4 supported?':   supported_lat,
            })

lr_table = pd.DataFrame(lr_rows)
print('H1-4 — Logistic Regression Hypothesis Conclusion Table')
print('=' * 90)
display(
    lr_table.style
    .set_caption('Table H1-4: Logistic Regression — Statistical Conclusions by Experiment & Comparison')
    .applymap(lambda v: 'color: #C62828; font-weight: bold' if v == 'Reject H₀' else
              ('color: #1565C0; font-weight: bold' if 'Fail' in str(v) else ''),
              subset=['Decision'])
    .applymap(lambda v: 'color: #C62828' if 'CONTRADICTED' in str(v) or v == 'NO' else
              ('color: #2E7D32' if 'YES' in str(v) else
               ('color: #E65100' if 'tied' in str(v).lower() or 'Partially' in str(v) else '')),
              subset=['H1-4 supported?'])
    .set_properties(**{'font-size': '11px', 'text-align': 'left'})
    .hide(axis='index')
)

lr_table.to_csv(os.path.join(STAT_DIR, 'table_H1-4_LR_conclusions_154class.csv'), index=False)
print(f'\n✅ Saved → results/statistical/table_H1-4_LR_conclusions_154class.csv')

In [ ]:
# ── Cell 16G: Master summary conclusion table ────────────────────────────
# One row per hypothesis per experiment — the dissertation-ready overview.

summary_rows = []

EXPERIMENT_ORDER = ['154-class']

for exp in EXPERIMENT_ORDER:
    if exp not in preds and exp not in latency:
        continue

    # H0G — Cochran's Q
    q_r   = cochran_df[cochran_df['Experiment'] == exp]
    kw_r  = kw_df[kw_df['Experiment'] == exp]
    q_sig  = (q_r['Significant (α=0.05)'] == 'YES').any()  if len(q_r)  else False
    kw_sig = (kw_r['Significant (α=0.05)'] == 'YES').any() if len(kw_r) else False

    summary_rows.append({
        'Experiment': exp, 'Hypothesis': 'H0G',
        'Focal model': 'All models',
        'Accuracy test': "Cochran's Q",
        'Acc decision': 'Reject H₀' if q_sig  else 'Fail to Reject H₀',
        'Latency test': 'Kruskal-Wallis',
        'Lat decision': 'Reject H₀' if kw_sig else 'Fail to Reject H₀',
        'Overall verdict': ('H₀G rejected — significant differences exist'
                            if q_sig or kw_sig else
                            'H₀G not rejected — models statistically equivalent'),
        'Key finding': ('Classifiers differ significantly in accuracy and/or speed'
                        if q_sig or kw_sig else 'No significant differences detected'),
    })

    # H1-1 CIF
    cif_in = 'CIF' in preds.get(exp, {})
    if cif_in:
        cif_acc = accuracy_score(*preds[exp]['CIF'])
        best    = all(cif_acc >= accuracy_score(*preds[exp].get(m, ([], []))) for m in preds[exp] if m != 'CIF')
        cif_med = np.median(latency.get(exp, {}).get('CIF', [np.nan]))
        cif_slowest = all(cif_med >= np.median(latency.get(exp, {}).get(m, [0]))
                          for m in latency.get(exp, {}) if m != 'CIF')
        verdict = ('Reject H1-1 — CIF worst on both accuracy and latency'
                   if not best and cif_slowest else
                   'Reject H1-1 — CIF not highest accuracy' if not best else
                   'Support H1-1')
        summary_rows.append({
            'Experiment': exp, 'Hypothesis': 'H1-1',
            'Focal model': 'CIF',
            'Accuracy test': "McNemar's (pairwise)",
            'Acc decision': 'Reject H₀' if best else 'Fail to Reject H₀ (CIF not best)',
            'Latency test': 'Mann-Whitney U',
            'Lat decision': 'Reject H₀ (CIF slowest)' if cif_slowest else 'Fail to Reject H₀',
            'Overall verdict': verdict,
            'Key finding': f'CIF acc={cif_acc*100:.1f}%, median latency={cif_med:.0f}ms',
        })

    # H1-2 InceptionTime
    it_in  = 'InceptionTime' in preds.get(exp, {})
    cif_in2 = 'CIF' in preds.get(exp, {})
    if it_in and cif_in2:
        r = get_sig_pairs(mcnemar_df, exp, 'InceptionTime', 'CIF')
        it_acc  = accuracy_score(*preds[exp]['InceptionTime'])
        cif_acc2 = accuracy_score(*preds[exp]['CIF'])
        it_med  = np.median(latency.get(exp, {}).get('InceptionTime', [np.nan]))
        cif_med2 = np.median(latency.get(exp, {}).get('CIF', [np.nan]))
        acc_conf = it_acc > cif_acc2 and (r is not None)
        lat_conf = it_med > cif_med2  # H1-2 predicts IT slower
        verdict  = ('Partially support H1-2 — acc confirmed, lat contradicted (IT faster)'
                    if acc_conf and not lat_conf else
                    'Support H1-2' if acc_conf and lat_conf else
                    'Reject H1-2')
        summary_rows.append({
            'Experiment': exp, 'Hypothesis': 'H1-2',
            'Focal model': 'InceptionTime',
            'Accuracy test': "McNemar's vs CIF",
            'Acc decision': 'Reject H₀ (IT better)' if acc_conf else 'Fail to Reject H₀',
            'Latency test': 'Mann-Whitney U vs CIF',
            'Lat decision': ('Reject H₀ (IT slower — as predicted)' if lat_conf else
                             'Reject H₀ (IT faster — contradicts H1-2)'),
            'Overall verdict': verdict,
            'Key finding': f'IT acc={it_acc*100:.1f}% vs CIF {cif_acc2*100:.1f}%; IT={it_med:.1f}ms vs CIF={cif_med2:.0f}ms',
        })

    # H1-3 Random Forest
    rf_in = 'Random Forest' in preds.get(exp, {})
    if rf_in:
        rf_acc = accuracy_score(*preds[exp]['Random Forest'])
        it_acc2 = accuracy_score(*preds[exp].get('InceptionTime', ([], []))) if 'InceptionTime' in preds.get(exp, {}) else None
        cif_acc3 = accuracy_score(*preds[exp].get('CIF', ([], []))) if 'CIF' in preds.get(exp, {}) else None
        rf_lt_it  = (rf_acc < it_acc2)  if it_acc2  else None
        rf_gt_cif = (rf_acc > cif_acc3) if cif_acc3 else None
        rf_med    = np.median(latency.get(exp, {}).get('Random Forest', [np.nan]))
        cif_med3  = np.median(latency.get(exp, {}).get('CIF', [np.nan]))
        it_med2   = np.median(latency.get(exp, {}).get('InceptionTime', [np.nan]))
        rf_faster_cif = rf_med < cif_med3
        rf_faster_it  = rf_med < it_med2
        verdict = 'Partially support H1-3'
        key = (f'RF acc={rf_acc*100:.1f}%; RF<IT: {rf_lt_it}, RF>CIF: {rf_gt_cif}; '
               f'RF faster than CIF: {rf_faster_cif}, faster than IT: {rf_faster_it}')
        summary_rows.append({
            'Experiment': exp, 'Hypothesis': 'H1-3',
            'Focal model': 'Random Forest',
            'Accuracy test': "McNemar's (pairwise)",
            'Acc decision': f'RF<IT: {rf_lt_it}, RF>CIF: {rf_gt_cif}',
            'Latency test': 'Mann-Whitney U',
            'Lat decision': f'Faster than CIF: {rf_faster_cif}, faster than IT: {rf_faster_it}',
            'Overall verdict': verdict,
            'Key finding': key,
        })

    # H1-4 Logistic Regression
    lr_in = 'Logistic Regression' in preds.get(exp, {})
    if lr_in:
        lr_acc  = accuracy_score(*preds[exp]['Logistic Regression'])
        all_acc = {m: accuracy_score(*preds[exp][m]) for m in preds[exp]}
        lr_lowest = all(lr_acc <= v for m, v in all_acc.items() if m != 'Logistic Regression')
        lr_med    = np.median(latency.get(exp, {}).get('Logistic Regression', [np.nan]))
        all_med   = {m: np.median(latency.get(exp, {}).get(m, [np.inf])) for m in latency.get(exp, {})}
        lr_fastest = all(lr_med <= v for m, v in all_med.items() if m != 'Logistic Regression')
        verdict = ('Partially support H1-4 — efficiency confirmed, not lowest accuracy'
                   if lr_fastest and not lr_lowest else
                   'Support H1-4' if lr_fastest and lr_lowest else
                   'Reject H1-4')
        summary_rows.append({
            'Experiment': exp, 'Hypothesis': 'H1-4',
            'Focal model': 'Logistic Regression',
            'Accuracy test': "McNemar's (pairwise)",
            'Acc decision': 'LR lowest accuracy' if lr_lowest else 'LR NOT lowest accuracy',
            'Latency test': 'Mann-Whitney U',
            'Lat decision': 'LR fastest' if lr_fastest else 'LR not fastest',
            'Overall verdict': verdict,
            'Key finding': f'LR acc={lr_acc*100:.1f}%, median={lr_med:.1f}ms; lowest acc: {lr_lowest}, fastest: {lr_fastest}',
        })

master_table = pd.DataFrame(summary_rows)

print('MASTER HYPOTHESIS CONCLUSION TABLE')
print('=' * 90)
display(
    master_table.style
    .set_caption('Master Statistical Conclusion Table — All Hypotheses × All Experiments')
    .applymap(lambda v: 'color: #C62828; font-weight: bold' if 'Reject H₀' in str(v) and 'Fail' not in str(v) else
              ('color: #1565C0; font-weight: bold' if 'Fail' in str(v) else ''),
              subset=['Acc decision', 'Lat decision'])
    .applymap(lambda v: 'background-color: #FFEBEE; font-weight: bold' if 'Reject' in str(v) and 'Partially' not in str(v) else
              ('background-color: #FFF8E1' if 'Partially' in str(v) else
               ('background-color: #E8F5E9' if 'Support' in str(v) and 'Partially' not in str(v) else '')),
              subset=['Overall verdict'])
    .set_properties(**{'font-size': '11px', 'text-align': 'left'})
    .hide(axis='index')
)

master_table.to_csv(os.path.join(STAT_DIR, 'table_MASTER_hypothesis_conclusions_154class.csv'), index=False)
print(f'\n✅ Saved → results/statistical/table_MASTER_hypothesis_conclusions_154class.csv')

In [ ]:
# ── Cell 16H: Updated final summary with all new outputs ─────────────────

print('=' * 80)
print('  ALL OUTPUTS — results/statistical/ (154-class)')
print('=' * 80)

all_files = [
    # Raw test results
    ('bootstrap_confidence_intervals_154class.csv',   'Bootstrap 95% CI — Accuracy & F1 per model×experiment'),
    ('global_cochrans_q_accuracy_154class.csv',       "Cochran's Q global accuracy test"),
    ('global_kruskal_wallis_latency_154class.csv',    'Kruskal-Wallis global latency test'),
    ('pairwise_mcnemar_accuracy_154class.csv',        "McNemar's pairwise + Cohen's h + Bonferroni"),
    ('pairwise_mannwhitney_latency_154class.csv',     "Mann-Whitney U pairwise + Cliff's delta + Bonferroni"),
    ('hypothesis_verdicts_154class.csv',              'Programmatic verdict per RQ per experiment'),
    # Per-model conclusion tables
    ('table_H0G_global_hypothesis_154class.csv',      'H0G — Global hypothesis structured conclusion'),
    ('table_H1-1_CIF_conclusions_154class.csv',       'H1-1 — CIF conclusion table (test × experiment)'),
    ('table_H1-2_InceptionTime_conclusions_154class.csv', 'H1-2 — InceptionTime conclusion table'),
    ('table_H1-3_RF_conclusions_154class.csv',        'H1-3 — Random Forest conclusion table'),
    ('table_H1-4_LR_conclusions_154class.csv',        'H1-4 — Logistic Regression conclusion table'),
    ('table_MASTER_hypothesis_conclusions_154class.csv', 'MASTER — All hypotheses × all experiments'),
    # Figures
    ('effect_size_summary_154class.png',              "Effect size forest plot (Cohen's h + Cliff's delta)"),
    ('ci_forest_plot_f1_154class.png',               'Bootstrap CI forest plot — Macro F1'),
    ('mcnemar_heatmap_*.png',               'p-value heatmaps per experiment'),
]

print(f'  {"File":<55} Description')
print(f'  {"-"*54} {"-"*35}')
for fname, desc in all_files:
    fpath = os.path.join(STAT_DIR, fname)
    exists = '✅' if (os.path.exists(fpath) or '*' in fname) else '⚠️ '
    print(f'  {exists}  {fname:<53} {desc}')

print('\n── Table structure key ──')
print('  H0G : Global — Cochran\'s Q (accuracy) + Kruskal-Wallis (latency)')
print('  H1-1: CIF — McNemar\'s vs all others + Mann-Whitney U')
print('  H1-2: InceptionTime — McNemar\'s + Mann-Whitney U (vs CIF & others)')
print('  H1-3: Random Forest — McNemar\'s + Mann-Whitney U (vs IT & CIF)')
print('  H1-4: Logistic Regression — McNemar\'s + Mann-Whitney U (vs all)')
print('  MASTER: One row per hypothesis per experiment — dissertation overview')
print()
print('── Why no ANOVA ──')
print('  One-way ANOVA requires normally distributed, continuous DVs with')
print('  equal variances. Accuracy metrics are bounded proportions; latency')
print('  is right-skewed. Cochran\'s Q and Kruskal-Wallis are the correct')
print('  non-parametric alternatives and make no distributional assumptions.')